# Araseの電磁場データについて、64 Hzデータを用いる。

# FACの定義および磁場の衛星スピントーンの補正の付加

# データ保存先

In [1]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# Araseの電場・磁場データの取得

In [2]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330'

psp.erg.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', no_update=True, get_support_data=True)
psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', no_update=True, get_support_data=True)

print("--- Loaded tplot variables ---")
print(pt.tplot_names())

26-Jan-26 11:17:08: del_data: No valid tplot variables found, returning
26-Jan-26 11:17:08: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/pwe/efd/l2/E64Hz/2022/09/erg_pwe_efd_l2_E64Hz_dsi_20220901_v01_02.cdf


 
 
**************************************************************************
['Exploration of Energization and Radiation in Geospace (ERG) Plasma Wave Experiment (PWE) Electric Field Detector (EFD) Level 2 waveform data in DSI Coordinate System']

Information about ERG PWE EFD

PI:  ['Yoshiya Kasahara']
Affiliation:  ['Kanazawa University']

RoR of ERG project common: https://ergsc.isee.nagoya-u.ac.jp/data_info/rules_of_the_road.shtml.en
RoR of PWE/EFD: https://ergsc.isee.nagoya-u.ac.jp/mw/index.php/ErgSat/Pwe/Efd

Contact: erg_pwe_info at isee.nagoya-u.ac.jp
**************************************************************************


26-Jan-26 11:17:11: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/mgf/l2/64hz/2022/09/erg_mgf_l2_64hz_dsi_2022090121_v04.05.cdf
26-Jan-26 11:17:11: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/mgf/l2/64hz/2022/09/erg_mgf_l2_64hz_dsi_2022090122_v04.05.cdf
26-Jan-26 11:17:11: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/mgf/l2/64hz/2022/09/erg_mgf_l2_64hz_dsi_2022090123_v04.05.cdf


 
**************************************************************************
['Exploration of Energization and Radiation in Geospace (ERG) Magnetic Field Experiment (MGF) Level 2 64 Hz resolution magnetic field data']

Information about ERG MGF

PI:  ['Ayako Matsuoka']
Affiliation:  ['Data Analysis Center for Geomagnetism and Space Magnetism, Graduate School of Science, Kyoto University, Kitashirakawa-Oiwake Cho, Sakyo-ku Kyoto 606-8502, Japan']

RoR of ERG project common: https://ergsc.isee.nagoya-u.ac.jp/data_info/rules_of_the_road.shtml.en
RoR of MGF L2: https://ergsc.isee.nagoya-u.ac.jp/mw/index.php/ErgSat/Mgf
Contact: erg_mgf_info at isee.nagoya-u.ac.jp
**************************************************************************
--- Loaded tplot variables ---
0 : erg_pwe_efd_l2_E64Hz_dsi_Epoch
1 : erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform
2 : erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform
3 : erg_pwe_efd_l2_E64Hz_dsi_time_offsets
4 : erg_pwe_efd_l2_E64Hz_dsi_ti_corrected
5 : erg_pwe_efd_l2_E64Hz_d

In [3]:
import xarray as xr
import numpy as np

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]

B64_data_dsi    = pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi_quality_flag   = pt.data_quants['erg_mgf_l2_quality_64hz'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

In [4]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [5]:
ds_B64_dsi_segs = split_by_gap(ds_B64_dsi, gap_thr=np.timedelta64(63, 'ms'))
print(len(ds_B64_dsi_segs))

1


In [6]:
ds_B64_dsi_seg0 = ds_B64_dsi_segs[0]
print(ds_B64_dsi_seg0)

<xarray.Dataset> Size: 22MB
Dimensions:    (time: 691238)
Coordinates:
  * time       (time) datetime64[ns] 6MB 2022-09-01T21:00:00.001522176 ... 20...
Data variables:
    B64_dsi_x  (time) float64 6MB 993.2 993.4 993.3 993.3 ... 161.9 161.8 161.9
    B64_dsi_y  (time) float64 6MB 224.6 224.4 224.5 ... -23.5 -23.54 -23.61
    B64_dsi_z  (time) float64 6MB 113.5 113.3 113.3 ... -148.3 -148.3 -148.2


# 磁場のスピントーン除去 [Imajo et al., 2021]

In [7]:
da_mgf_spin_phase_deg           = pt.data_quants['erg_mgf_l2_spin_phase_64hz'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
da_mgf_spin_phase_deg_interp    = da_mgf_spin_phase_deg.interp(time=ds_B64_dsi.time)
da_mgf_spin_phase_rad_interp    = np.deg2rad(da_mgf_spin_phase_deg_interp)

In [8]:
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.erg_mgf_spintone_rm as emsr
importlib.reload(emsr)
os.chdir('./KAW_observation')
print(os.getcwd())

B_clean_ndarray, B_spt_ndarray, params  = emsr.remove_spintone_3comp(
    time=ds_B64_dsi_seg0.time.values,
    Bx=ds_B64_dsi_seg0['B64_dsi_x'].values,
    By=ds_B64_dsi_seg0['B64_dsi_y'].values,
    Bz=ds_B64_dsi_seg0['B64_dsi_z'].values,
    phase_rad=da_mgf_spin_phase_rad_interp.values,
    min_points=64.*3./2.
)

ds_B64_dsi_seg0_spt      = xr.Dataset({
    'B64_dsi_x_spt':    ('time', B_spt_ndarray[:, 0]),
    'B64_dsi_y_spt':    ('time', B_spt_ndarray[:, 1]),
    'B64_dsi_z_spt':    ('time', B_spt_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

ds_B64_dsi_seg0_clean    = xr.Dataset({
    'B64_dsi_x_clean':    ('time', B_clean_ndarray[:, 0]),
    'B64_dsi_y_clean':    ('time', B_clean_ndarray[:, 1]),
    'B64_dsi_z_clean':    ('time', B_clean_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

print(ds_B64_dsi_seg0_spt)
print(ds_B64_dsi_seg0_clean)

/home/satanka/Documents/observation_workspace
/home/satanka/Documents/observation_workspace/KAW_observation
<xarray.Dataset> Size: 22MB
Dimensions:        (time: 691238)
Coordinates:
  * time           (time) datetime64[ns] 6MB 2022-09-01T21:00:00.001522176 .....
Data variables:
    B64_dsi_x_spt  (time) float64 6MB -235.7 -220.8 -205.9 ... 0.4078 0.4274
    B64_dsi_y_spt  (time) float64 6MB -55.89 -52.47 -49.07 ... -1.676 -1.671
    B64_dsi_z_spt  (time) float64 6MB -26.64 -24.94 -23.24 ... -0.03839 -0.03799
<xarray.Dataset> Size: 22MB
Dimensions:          (time: 691238)
Coordinates:
  * time             (time) datetime64[ns] 6MB 2022-09-01T21:00:00.001522176 ...
Data variables:
    B64_dsi_x_clean  (time) float64 6MB 1.229e+03 1.214e+03 ... 161.4 161.5
    B64_dsi_y_clean  (time) float64 6MB 280.5 276.9 273.6 ... -21.86 -21.94
    B64_dsi_z_clean  (time) float64 6MB 140.1 138.2 136.6 ... -148.2 -148.2


# 電場データと磁場データの時間を合わせる

In [9]:
E64_data_dsi_x  = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_y  = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_quality_flag = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_quality_flag'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

# QF = 0 の時間を抽出
bad_times = E64_data_dsi_quality_flag.time.where(E64_data_dsi_quality_flag != 0, drop=True)

E64_data_dsi_x_qf = E64_data_dsi_x.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_x.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)
E64_data_dsi_y_qf = E64_data_dsi_y.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_y.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)

ds_E64_dsi_xy   = xr.Dataset({
    'E64_dsi_x': E64_data_dsi_x_qf,
    'E64_dsi_y': E64_data_dsi_y_qf
})
ds_E64_dsi_xy   = ds_E64_dsi_xy.dropna(dim='time', how='all')

In [10]:
ds_E64_dsi_xy_segs = split_by_gap(ds_E64_dsi_xy, gap_thr=np.timedelta64(63, 'ms'))
print(ds_E64_dsi_xy_segs)

[<xarray.Dataset> Size: 7kB
Dimensions:    (time: 452)
Coordinates:
  * time       (time) datetime64[ns] 4kB 2022-09-01T21:00:00.002088960 ... 20...
Data variables:
    E64_dsi_x  (time) float32 2kB -0.8878 -0.8309 -0.79 ... 0.2553 0.1913 0.1436
    E64_dsi_y  (time) float32 2kB 0.4577 0.4531 0.4324 ... 1.917 1.879 1.836, <xarray.Dataset> Size: 3MB
Dimensions:    (time: 209024)
Coordinates:
  * time       (time) datetime64[ns] 2MB 2022-09-01T21:05:41.516568832 ... 20...
Data variables:
    E64_dsi_x  (time) float32 836kB -0.2142 -0.2574 -0.2998 ... 0.714 0.7033
    E64_dsi_y  (time) float32 836kB 0.8039 0.7877 0.7529 ... 0.6613 0.657 0.6525, <xarray.Dataset> Size: 7kB
Dimensions:    (time: 416)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2022-09-01T22:05:39.123286016 ... 20...
Data variables:
    E64_dsi_x  (time) float32 2kB 1.032 1.059 1.076 1.104 ... 1.413 1.407 1.434
    E64_dsi_y  (time) float32 2kB 0.7673 0.7529 0.7187 ... -0.6292 -0.429, <xarray.Dataset> Size: 3MB
Dime

In [11]:
def make_ds_EB_func(ds_E_xy, E_vars, ds_B_xyz, B_vars, output_vars):
    time_base   = ds_E_xy.time
    ds_B_xyz_interp  = ds_B_xyz.interp(time=time_base, method='linear')

    da_Ex   = ds_E_xy[E_vars[0]]
    da_Ey   = ds_E_xy[E_vars[1]]
    da_Bx   = ds_B_xyz_interp[B_vars[0]]
    da_By   = ds_B_xyz_interp[B_vars[1]]
    da_Bz   = ds_B_xyz_interp[B_vars[2]]

    da_Ez   = xr.where(np.abs(da_Bz) > 1E-2, -(da_Ex * da_Bx + da_Ey * da_By) / da_Bz, np.nan)

    ds_EB   = xr.Dataset({
        output_vars[0]: da_Ex,
        output_vars[1]: da_Ey,
        output_vars[2]: da_Ez,
        output_vars[3]: da_Bx,
        output_vars[4]: da_By,
        output_vars[5]: da_Bz,
    })

    ds_EB   = ds_EB.dropna(dim='time', how='any')

    return ds_EB

In [12]:
E64_xy_vars     = ['E64_dsi_x', 'E64_dsi_y']
B64_xyz_vars    = ['B64_dsi_x_clean', 'B64_dsi_y_clean', 'B64_dsi_z_clean']
EB64_vars       = ['E64_dsi_x', 'E64_dsi_y', 'E64_dsi_z', 'B64_dsi_x', 'B64_dsi_y', 'B64_dsi_z']

In [13]:
ds_EB64_dsi_segs    = []

for count in range(len(ds_E64_dsi_xy_segs)):
    ds_     = make_ds_EB_func(ds_E64_dsi_xy_segs[count], E64_xy_vars, ds_B64_dsi_seg0_clean, B64_xyz_vars, EB64_vars)
    ds_EB64_dsi_segs.append(ds_)

print(ds_EB64_dsi_segs)

[<xarray.Dataset> Size: 22kB
Dimensions:    (time: 452)
Coordinates:
  * time       (time) datetime64[ns] 4kB 2022-09-01T21:00:00.002088960 ... 20...
Data variables:
    E64_dsi_x  (time) float32 2kB -0.8878 -0.8309 -0.79 ... 0.2553 0.1913 0.1436
    E64_dsi_y  (time) float32 2kB 0.4577 0.4531 0.4324 ... 1.917 1.879 1.836
    E64_dsi_z  (time) float64 4kB 6.871 6.391 6.071 ... -6.122 -5.478 -4.968
    B64_dsi_x  (time) float64 4kB 1.228e+03 1.214e+03 ... 1.228e+03 1.224e+03
    B64_dsi_y  (time) float64 4kB 280.3 276.8 273.5 270.1 ... 280.8 279.7 278.7
    B64_dsi_z  (time) float64 4kB 140.1 138.2 136.5 134.9 ... 139.3 138.8 138.4, <xarray.Dataset> Size: 10MB
Dimensions:    (time: 209017)
Coordinates:
  * time       (time) datetime64[ns] 2MB 2022-09-01T21:05:41.516568832 ... 20...
Data variables:
    E64_dsi_x  (time) float32 836kB -0.2142 -0.2574 -0.2998 ... 0.714 0.7033
    E64_dsi_y  (time) float32 836kB 0.8039 0.7877 0.7529 ... 0.6613 0.657 0.6525
    E64_dsi_z  (time) float64 2MB 

In [14]:
path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330'
#os.makedirs(path_base_save_plot, exist_ok=True)

In [15]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T21:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E64_dsi_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E64_dsi_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E64_dsi_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B64_dsi_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B64_dsi_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B64_dsi_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (DSI)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (DSI)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (DSI)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (DSI)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (DSI)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (DSI)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_dsi'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB64_dsi_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

# FAC座標系を定義、DSI座標系 -> FAC座標系変換行列の作成

# FAC座標系の定義
- z軸は、背景磁場$B_{0}$の単位ベクトルで与える。
- x軸は、反地球方向かつz軸と垂直な単位ベクトルで与える。 ($E_{x}$: Toroidal component, $B_{x}$: Poloidal component)
- y軸は、z軸とx軸の外積で与える。 ($E_{y}$: Poloidal component, $B_{y}$: Toroidal component)

In [16]:
psp.erg.orb(trange=time_range, level='l2', datatype='def', no_update=True)
da_Arase_pos_gsm    = pt.data_quants['erg_orb_l2_pos_gsm']
da_Arase_pos_gsm    = da_Arase_pos_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

da_Arase_pos_unit_gsm   = da_Arase_pos_gsm / np.sqrt((da_Arase_pos_gsm * da_Arase_pos_gsm).sum(dim='v_dim'))

pt.store_data('Arase_pos_unit_gsm', data={'x': da_Arase_pos_unit_gsm.time, 'y': da_Arase_pos_unit_gsm.data})
psp.cotrans(name_in='Arase_pos_unit_gsm', name_out='Arase_pos_unit_j2000', coord_in='gsm', coord_out='j2000')
psp.erg.erg_cotrans(in_name='Arase_pos_unit_j2000', out_name='Arase_pos_unit_dsi', in_coord='j2000', out_coord='dsi')

da_Arase_pos_unit_dsi   = pt.data_quants['Arase_pos_unit_dsi']

26-Jan-26 11:17:18: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/orb/def/2022/erg_orb_l2_20220901_v05.cdf
26-Jan-26 11:17:18: ['gsm', 'gse', 'gei', 'j2000']
26-Jan-26 11:17:18: Running transformation: subgsm2gse
26-Jan-26 11:17:18: Running transformation: subgse2gei
26-Jan-26 11:17:18: Running transformation: subgei2j2000
26-Jan-26 11:17:18: Setting coordinate system for Arase_pos_unit_j2000
26-Jan-26 11:17:18: Output variable: Arase_pos_unit_j2000
26-Jan-26 11:17:18: wildcard_expand: No match found for erg_att_sprate
26-Jan-26 11:17:18: Downloading remote index: https://ergsc.isee.nagoya-u.ac.jp/data/ergsc/satellite/erg/att/txt/


 
**************************************************************************
['Exploration of Energization and Radiation in Geospace (ERG) Level-2 orbit data']

Information about ERG orbit


RoR of ERG project common: https://ergsc.isee.nagoya-u.ac.jp/data_info/rules_of_the_road.shtml.en

Contact: erg-sc-core at isee.nagoya-u.ac.jp
**************************************************************************


26-Jan-26 11:17:19: File is current: /mnt/j/observation_data//ergsc/satellite/erg/att/txt/erg_att_l2_20220901_v03.txt
26-Jan-26 11:17:19: File is current: /mnt/j/observation_data//ergsc/satellite/erg/att/txt/erg_att_l2_20220902_v03.txt
26-Jan-26 11:17:20: Downloading remote index: https://ergsc.isee.nagoya-u.ac.jp/data/ergsc/satellite/erg/orb/def/2022/
26-Jan-26 11:17:26: File is current: /mnt/j/observation_data//ergsc/satellite/erg/orb/def/2022/erg_orb_l2_20220901_v05.cdf
26-Jan-26 11:17:26: File is current: /mnt/j/observation_data//ergsc/satellite/erg/orb/def/2022/erg_orb_l2_20220902_v05.cdf
26-Jan-26 11:17:26: tinterpol (linear) was applied to: erg_orb_l2_pos_gse-itrp
26-Jan-26 11:17:26: ['gse', 'gei', 'j2000']
26-Jan-26 11:17:26: Running transformation: subgse2gei
26-Jan-26 11:17:26: Running transformation: subgei2j2000
26-Jan-26 11:17:26: Setting coordinate system for sundir_j2000
26-Jan-26 11:17:26: Output variable: sundir_j2000


 
**************************************************************************
['Exploration of Energization and Radiation in Geospace (ERG) Level-2 orbit data']

Information about ERG orbit


RoR of ERG project common: https://ergsc.isee.nagoya-u.ac.jp/data_info/rules_of_the_road.shtml.en

Contact: erg-sc-core at isee.nagoya-u.ac.jp
**************************************************************************
J2000 --> DSI


In [17]:
background_time_sec = 100 #[sec]

In [18]:
time_width_B64          = (ds_B64_dsi_seg0_clean.time[10] - ds_B64_dsi_seg0_clean.time[9]) / np.timedelta64(1, 's')
ds_B_background         = ds_B64_dsi_seg0_clean.rolling(time=int(background_time_sec / time_width_B64), center=True).mean('time')

da_B_background         = ds_B_background.to_dataarray(dim='v_dim').T.dropna(dim='time', how='any')

print(da_B_background)

da_B_background_unit    = da_B_background / np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
da_B_background_unit    = da_B_background_unit.dropna(how='all', dim='time')

print(da_B_background_unit)
print(np.nanmin(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))
print(np.nanmax(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))

<xarray.DataArray (time: 684829, v_dim: 3)> Size: 16MB
array([[ 957.6148567 ,  212.97709288,  102.29056533],
       [ 957.57270421,  212.96580736,  102.28362211],
       [ 957.53285883,  212.95505886,  102.27698497],
       ...,
       [ 162.43063346,  -21.01998008, -147.86699149],
       [ 162.43031728,  -21.02022385, -147.86706555],
       [ 162.4300013 ,  -21.02045491, -147.86713532]], shape=(684829, 3))
Coordinates:
  * time     (time) datetime64[ns] 5MB 2022-09-01T21:00:50.050625152 ... 2022...
  * v_dim    (v_dim) object 24B 'B64_dsi_x_clean' ... 'B64_dsi_z_clean'
<xarray.DataArray (time: 684829, v_dim: 3)> Size: 16MB
array([[ 0.97088591,  0.21592862,  0.10370815],
       [ 0.97088656,  0.21592683,  0.10370575],
       [ 0.97088721,  0.21592506,  0.1037034 ],
       ...,
       [ 0.73611755, -0.09526021, -0.67011674],
       [ 0.73611665, -0.09526138, -0.67011756],
       [ 0.73611576, -0.0952625 , -0.67011838]], shape=(684829, 3))
Coordinates:
  * time     (time) datetime64[ns] 

In [19]:
da_Arase_pos_unit_dsi_interp    = da_Arase_pos_unit_dsi.interp(time=da_B_background_unit.time)

da_u_   = da_Arase_pos_unit_dsi_interp - (da_Arase_pos_unit_dsi_interp * da_B_background_unit).sum(dim='v_dim') * da_B_background_unit

da_e_z_FAC_inDSI    = da_B_background_unit.drop_attrs()
da_e_x_FAC_inDSI    = (da_u_ / np.sqrt((da_u_ * da_u_).sum(dim='v_dim'))).drop_attrs()
da_e_y_FAC_inDSI    = (xr.apply_ufunc(np.cross, da_e_z_FAC_inDSI, da_e_x_FAC_inDSI, input_core_dims=[['v_dim'], ['v_dim']], output_core_dims=[['v_dim']], vectorize=True)).drop_attrs()

print(da_e_x_FAC_inDSI)
print('')
print(da_e_y_FAC_inDSI)
print('')
print(da_e_z_FAC_inDSI)

<xarray.DataArray (time: 684829, v_dim: 3)> Size: 16MB
array([[-0.17770851,  0.93957358, -0.29261097],
       [-0.17770743,  0.93957379, -0.29261095],
       [-0.17770637,  0.93957402, -0.29261085],
       ...,
       [-0.14266017,  0.94597006, -0.29118502],
       [-0.14266045,  0.9459696 , -0.29118637],
       [-0.14266074,  0.94596916, -0.29118766]], shape=(684829, 3))
Coordinates:
  * time     (time) datetime64[ns] 5MB 2022-09-01T21:00:50.050625152 ... 2022...
  * v_dim    (v_dim) object 24B 'B64_dsi_x_clean' ... 'B64_dsi_z_clean'

<xarray.DataArray (time: 684829, v_dim: 3)> Size: 16MB
array([[-0.16062453,  0.26566205,  0.9505911 ],
       [-0.16062176,  0.26566276,  0.95059137],
       [-0.16061904,  0.26566337,  0.95059166],
       ...,
       [ 0.66164872,  0.30994537,  0.68275532],
       [ 0.66164966,  0.30994641,  0.68275394],
       [ 0.66165059,  0.30994741,  0.68275259]], shape=(684829, 3))
Coordinates:
  * time     (time) datetime64[ns] 5MB 2022-09-01T21:00:50.050625152 .

In [20]:
R_FAC_to_DSI = xr.concat(
    [da_e_x_FAC_inDSI, da_e_y_FAC_inDSI, da_e_z_FAC_inDSI],
    dim='axis'
)
R_FAC_to_DSI = R_FAC_to_DSI.assign_coords(axis=['x_FAC', 'y_FAC', 'z_FAC']).assign_coords(v_dim=np.arange(3))

R_DSI_to_FAC = R_FAC_to_DSI.transpose('time', 'v_dim', 'axis')

print(R_FAC_to_DSI)
print('')
print(R_DSI_to_FAC)

<xarray.DataArray (axis: 3, time: 684829, v_dim: 3)> Size: 49MB
array([[[-0.17770851,  0.93957358, -0.29261097],
        [-0.17770743,  0.93957379, -0.29261095],
        [-0.17770637,  0.93957402, -0.29261085],
        ...,
        [-0.14266017,  0.94597006, -0.29118502],
        [-0.14266045,  0.9459696 , -0.29118637],
        [-0.14266074,  0.94596916, -0.29118766]],

       [[-0.16062453,  0.26566205,  0.9505911 ],
        [-0.16062176,  0.26566276,  0.95059137],
        [-0.16061904,  0.26566337,  0.95059166],
        ...,
        [ 0.66164872,  0.30994537,  0.68275532],
        [ 0.66164966,  0.30994641,  0.68275394],
        [ 0.66165059,  0.30994741,  0.68275259]],

       [[ 0.97088591,  0.21592862,  0.10370815],
        [ 0.97088656,  0.21592683,  0.10370575],
        [ 0.97088721,  0.21592506,  0.1037034 ],
        ...,
        [ 0.73611755, -0.09526021, -0.67011674],
        [ 0.73611665, -0.09526138, -0.67011756],
        [ 0.73611576, -0.0952625 , -0.67011838]]], shape=(3,

In [21]:
da_e_x_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=0)
da_e_y_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=1)
da_e_z_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=2)

In [22]:
#import os
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#mpl.rcParams['font.size'] = 15
#
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330/coordinate"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#def setup_ax(ax, xlabel, ylabel, title):
#    ax.axhline(0, color='k', linewidth=0.5)
#    ax.axvline(0, color='k', linewidth=0.5)
#    ax.set_aspect('equal', adjustable='box')
#    ax.set_xlim(-1, 1)
#    ax.set_ylim(-1, 1)
#    ax.set_xlabel(xlabel)
#    ax.set_ylabel(ylabel)
#    ax.minorticks_on()
#    ax.grid(True, which='both', linestyle=':')
#    ax.set_title(title)
#
#def plot_fac_dsi_frame(it, frame_idx, save_dir):
#    ex = da_e_x_DSI_inFAC.isel(time=it)
#    ey = da_e_y_DSI_inFAC.isel(time=it)
#    ez = da_e_z_DSI_inFAC.isel(time=it)
#
#    # FAC 成分
#    ex_x = ex.sel(axis='x_FAC').item()
#    ex_y = ex.sel(axis='y_FAC').item()
#    ex_z = ex.sel(axis='z_FAC').item()
#
#    ey_x = ey.sel(axis='x_FAC').item()
#    ey_y = ey.sel(axis='y_FAC').item()
#    ey_z = ey.sel(axis='z_FAC').item()
#
#    ez_x = ez.sel(axis='x_FAC').item()
#    ez_y = ez.sel(axis='y_FAC').item()
#    ez_z = ez.sel(axis='z_FAC').item()
#
#    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
#
#    # ------------- (x, y) plane -------------
#    ax = axs[0]
#    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
#              color='k',   label='FAC-x')
#    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
#              color='gray', label='FAC-y')
#    ax.quiver(0, 0, 0, 0, angles='xy', scale_units='xy', scale=1,
#              color='purple', label='FAC-z', linewidth=0)
#
#    ax.quiver(0, 0, ex_x, ex_y, angles='xy', scale_units='xy', scale=1,
#              color='r', label='DSI-x')
#    ax.quiver(0, 0, ey_x, ey_y, angles='xy', scale_units='xy', scale=1,
#              color='b', label='DSI-y')
#    ax.quiver(0, 0, ez_x, ez_y, angles='xy', scale_units='xy', scale=1,
#              color='g', label='DSI-z')
#
#    setup_ax(ax, 'FAC-x (Radial)', 'FAC-y (Longitudinal)', '(x, y) plane')
#    ax.legend(loc='lower left')
#
#    # ------------- (x, z) plane -------------
#    ax = axs[1]
#    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
#              color='k',   label='FAC-x')
#    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
#              color='purple', label='FAC-z')
#
#    ax.quiver(0, 0, ex_x, ex_z, angles='xy', scale_units='xy', scale=1,
#              color='r', label='DSI-x')
#    ax.quiver(0, 0, ey_x, ey_z, angles='xy', scale_units='xy', scale=1,
#              color='b', label='DSI-y')
#    ax.quiver(0, 0, ez_x, ez_z, angles='xy', scale_units='xy', scale=1,
#              color='g', label='DSI-z')
#
#    setup_ax(ax, 'FAC-x (Radial)', 'FAC-z (Parallel)', '(x, z) plane')
#
#    # ------------- (y, z) plane -------------
#    ax = axs[2]
#    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
#              color='gray',   label='FAC-y')
#    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
#              color='purple', label='FAC-z')
#
#    ax.quiver(0, 0, ex_y, ex_z, angles='xy', scale_units='xy', scale=1,
#              color='r', label='DSI-x')
#    ax.quiver(0, 0, ey_y, ey_z, angles='xy', scale_units='xy', scale=1,
#              color='b', label='DSI-y')
#    ax.quiver(0, 0, ez_y, ez_z, angles='xy', scale_units='xy', scale=1,
#              color='g', label='DSI-z')
#
#    setup_ax(ax, 'FAC-y (Longitudinal)', 'FAC-z (Parallel)', '(y, z) plane')
#
#    fig.suptitle(str(da_e_x_DSI_inFAC.time.values[it]))
#    plt.tight_layout()
#
#    # ファイル名：time index をゼロ埋め
#    fname = os.path.join(save_dir, f"coord_{frame_idx:06d}.png")
#    fig.savefig(fname, dpi=150)
#    plt.close(fig)


In [23]:
#from concurrent.futures import ProcessPoolExecutor, as_completed
#import multiprocessing as mp
#from tqdm import tqdm
#import numpy as np
#
## --- 1) タスク作成 ---
#n_time = da_e_x_DSI_inFAC.sizes['time']
#step = 64 * 60   # 64Hz × 60 sec
#
#tasks = []
#frame_idx = 0
#for it in range(0, n_time, step):
#    tasks.append((it, frame_idx))
#    frame_idx += 1
#
#print("num frames:", len(tasks))
#
#
## --- 2) worker ---
#def worker(args):
#    it, frame_idx, save_path = args
#    plot_fac_dsi_frame(it, frame_idx, save_path)
#    return frame_idx
#
#
## --- 3) 並列 + tqdm ---
#save_path = path_base_save_plot
#n_workers = max(1, mp.cpu_count() - 1)
#
#with ProcessPoolExecutor(max_workers=n_workers) as exe:
#    futures = [
#        exe.submit(worker, (it, idx, save_path))
#        for it, idx in tasks
#    ]
#
#    for f in tqdm(as_completed(futures), total=len(futures)):
#        _ = f.result()   # 例外を拾うため

# DSI座標系 -> FAC座標系変換の実行

In [24]:
ds_EB64_fac_segs = []

for ds_seg in ds_EB64_dsi_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_DSI_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['E64_dsi_x'], ds_seg['E64_dsi_y'], ds_seg['E64_dsi_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['B64_dsi_x'], ds_seg['B64_dsi_y'], ds_seg['B64_dsi_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'E64_fac_x',
            'y_FAC': 'E64_fac_y',
            'z_FAC': 'E64_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'B64_fac_x',
            'y_FAC': 'B64_fac_y',
            'z_FAC': 'B64_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds])
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EB64_fac_segs.append(ds_fac)


26-Jan-26 11:17:39: /tmp/ipykernel_2396/2549268582.py:22: PendingDeprecationWarning: The `dims` argument has been renamed to `dim`, and will be removed in the future. This renaming is taking place throughout xarray over the next few releases.
  E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)

26-Jan-26 11:17:39: /tmp/ipykernel_2396/2549268582.py:23: PendingDeprecationWarning: The `dims` argument has been renamed to `dim`, and will be removed in the future. This renaming is taking place throughout xarray over the next few releases.
  B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')



In [25]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T21:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E64_fac_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E64_fac_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E64_fac_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B64_fac_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B64_fac_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B64_fac_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB64_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

# 軌道データから、衛星速度(DSI)を導出

In [26]:
pos   = pt.data_quants['erg_orb_l2_pos_gse'].data   # (Nt,3)[R_E]
t_pos = pt.data_quants['erg_orb_l2_pos_gse'].time.values

R_E_m = 6.378137e6
t_s   = t_pos.astype('datetime64[ns]').astype(float) * 1e-9
v_gse = np.gradient(pos * R_E_m, t_s, axis=0)

v_sc_gse = xr.DataArray(
    data=v_gse, dims=('time','v_dim'),
    coords={'time': t_pos},
    attrs={'units':'m/s','desc':'$V_{sc}$ (GSE)'}
)
v_sc_gse.name = 'v_sc_gse'

print(v_sc_gse)

<xarray.DataArray 'v_sc_gse' (time: 28801, v_dim: 3)> Size: 346kB
array([[ 3734.1667,  -570.5   , -3352.5   ],
       [ 3737.4167,  -569.0417, -3352.9375],
       [ 3743.75  ,  -565.9167, -3353.8125],
       ...,
       [ 8061.229 ,  5398.75  ,  1246.0416],
       [ 8049.2915,  5424.1875,  1285.2084],
       [ 8043.396 ,  5436.875 ,  1304.9166]],
      shape=(28801, 3), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 230kB 2022-09-01 ... 2022-09-03
Dimensions without coordinates: v_dim
Attributes:
    units:    m/s
    desc:     $V_{sc}$ (GSE)


In [27]:
path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330"
)

In [28]:
#import matplotlib.pyplot as plt
#import datetime
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_gse_analysis  = v_sc_gse.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_gse_analysis.time, v_sc_gse_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (GSE)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (GSE)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (GSE)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_sc_gse_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_gse.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [29]:
import numpy as np, pytplot as pt, pyspedas as psp
from pyspedas.projects.erg.satellite.erg.common.cotrans.dsi2j2000 import dsi2j2000

def store64(name, t, y):
    pt.store_data(name, data={'x': t.astype('datetime64[ns]'),
                              'y': np.asarray(y, dtype=np.float64)})

# 64bitで登録
store64('v_sc_gse_64', v_sc_gse.time.values, v_sc_gse.data)

# 変換
psp.cotrans(name_in='v_sc_gse_64', name_out='v_sc_j2000_64', coord_in='gse', coord_out='j2000')
dsi2j2000(name_in='v_sc_j2000_64', name_out='v_sc_dsi_64', J20002DSI=True)

# 検証
def vnorm(name): dq=pt.data_quants[name]; return np.linalg.norm(dq.data,axis=1)
ng, nj, nd = vnorm('v_sc_gse_64'), vnorm('v_sc_j2000_64'), vnorm('v_sc_dsi_64')

def check(a,b,tag,thr_rel=1e-12):
    rel = np.abs(a-b)/np.maximum(a,1e-30)
    print(f"{tag}: rel mean={rel.mean():.3e}, max={rel.max():.3e}")
    assert np.nanmax(rel) < thr_rel, f"{tag} norm not preserved"

check(ng,nj,"GSE→J2000")
check(nj,nd,"J2000→DSI")

v_sc_j2000  = pt.data_quants['v_sc_j2000_64']
v_sc_dsi    = pt.data_quants['v_sc_dsi_64']

26-Jan-26 11:17:40: ['gse', 'gei', 'j2000']
26-Jan-26 11:17:40: Running transformation: subgse2gei
26-Jan-26 11:17:40: Running transformation: subgei2j2000
26-Jan-26 11:17:40: Setting coordinate system for v_sc_j2000_64
26-Jan-26 11:17:40: Output variable: v_sc_j2000_64
26-Jan-26 11:17:40: Downloading remote index: https://ergsc.isee.nagoya-u.ac.jp/data/ergsc/satellite/erg/orb/def/2022/
26-Jan-26 11:17:40: File is current: /mnt/j/observation_data//ergsc/satellite/erg/orb/def/2022/erg_orb_l2_20220831_v05.cdf
26-Jan-26 11:17:41: File is current: /mnt/j/observation_data//ergsc/satellite/erg/orb/def/2022/erg_orb_l2_20220901_v05.cdf
26-Jan-26 11:17:41: File is current: /mnt/j/observation_data//ergsc/satellite/erg/orb/def/2022/erg_orb_l2_20220902_v05.cdf
26-Jan-26 11:17:41: File is current: /mnt/j/observation_data//ergsc/satellite/erg/orb/def/2022/erg_orb_l2_20220903_v05.cdf
26-Jan-26 11:17:41: tinterpol (linear) was applied to: erg_orb_l2_pos_gse-itrp
26-Jan-26 11:17:41: ['gse', 'gei', 'j20

 
**************************************************************************
['Exploration of Energization and Radiation in Geospace (ERG) Level-2 orbit data']

Information about ERG orbit


RoR of ERG project common: https://ergsc.isee.nagoya-u.ac.jp/data_info/rules_of_the_road.shtml.en

Contact: erg-sc-core at isee.nagoya-u.ac.jp
**************************************************************************


26-Jan-26 11:17:42: Running transformation: subgei2j2000
26-Jan-26 11:17:42: Setting coordinate system for sundir_j2000
26-Jan-26 11:17:42: Output variable: sundir_j2000


J2000 --> DSI
GSE→J2000: rel mean=1.018e-16, max=6.454e-16
J2000→DSI: rel mean=nan, max=nan


In [30]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_j2000_analysis  = v_sc_j2000.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_j2000_analysis.time, v_sc_j2000_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (J2000)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (J2000)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (J2000)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_sc_j2000_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_j2000.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [31]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_dsi_analysis  = v_sc_dsi.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_dsi_analysis.time, v_sc_dsi_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (DSI)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (DSI)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (DSI)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_sc_dsi_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_dsi.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [32]:
R_interp    = R_DSI_to_FAC.interp(time=v_sc_dsi.time)

v_sc_fac    = xr.dot(v_sc_dsi, R_interp, dims='v_dim')
v_sc_fac    = v_sc_fac.dropna(dim='time', how='any')
print(v_sc_fac)

26-Jan-26 11:17:42: /tmp/ipykernel_2396/2540961186.py:3: PendingDeprecationWarning: The `dims` argument has been renamed to `dim`, and will be removed in the future. This renaming is taking place throughout xarray over the next few releases.
  v_sc_fac    = xr.dot(v_sc_dsi, R_interp, dims='v_dim')



<xarray.DataArray 'v_sc_dsi_64' (time: 1782, axis: 3)> Size: 43kB
array([[ 2260.63627806,  3077.93782182, -2144.36645352],
       [ 2256.42426605,  3075.87601564, -2144.56861728],
       [ 2252.65143147,  3073.80755823, -2144.680181  ],
       ...,
       [ -349.07425646,  1651.6494932 ,  -630.17593373],
       [ -350.29565195,  1651.19812903,  -629.38703474],
       [ -351.50748212,  1650.66153131,  -628.89537857]], shape=(1782, 3))
Coordinates:
  * time     (time) datetime64[ns] 14kB 2022-09-01T21:00:54 ... 2022-09-01T23...
  * axis     (axis) <U5 60B 'x_FAC' 'y_FAC' 'z_FAC'
Attributes:
    plot_options:  {'xaxis_opt': {'axis_label': '', 'crosshair': 'X', 'x_axis...
    data_att:      {'coord_sys': 'J2000'}


In [33]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_fac_analysis  = v_sc_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_sc_fac_analysis.time, np.sqrt(v_sc_fac_analysis.data[:, 0]**2E0 + v_sc_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sc}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sc_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# イオン流速($\approx$ MHD流速)、イオン温度$\rightarrow$イオン熱速度、電子温度$\rightarrow$ ion acoustic sppedの導出

In [34]:
pt.store_data('erg_mgf_l2_mag_64hz_background_dsi', data={'x': da_B_background.time, 'y': da_B_background.data})

True

In [35]:
import pyspedas as psp
import pytplot as pt

psp.erg.lepe(trange=time_range, datatype='3dflux', level='l2', no_update=True)
psp.erg.lepi(trange=time_range, datatype='3dflux', level='l2', no_update=True)
psp.erg.pwe_hfa(trange=time_range, level='l3', no_update=True)

psp.projects.erg.erg_lep_part_products(
    'erg_lepe_l2_3dflux_FEDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_background_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FPDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_background_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FHEDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_background_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FODU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_background_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

26-Jan-26 11:17:43: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/lepe/l2/3dflux/2022/09/erg_lepe_l2_3dflux_20220901_v04_01.cdf
26-Jan-26 11:17:48: Variable FEDU DEPEND_1 attribute FEDU_Energy has too many dimensions (3). Keeping extra dimensions (for now).
26-Jan-26 11:17:50: Variable Count_Rate DEPEND_1 attribute FEDU_Energy has too many dimensions (3). Keeping extra dimensions (for now).
26-Jan-26 11:17:51: Variable Count_Rate_BG DEPEND_1 attribute FEDU_Energy has too many dimensions (3). Keeping extra dimensions (for now).
26-Jan-26 11:17:53: erg_lepe_l2_3dflux_FEDU contains negative values; setting the z-axis to log scale will cause the negative values to be ignored on figures.


 
**************************************************************************
['Exploration of Energization and Radiation in Geospace (ERG) Low-Energy Particle experiments - electron analyzer (LEP-e) Level 2 3D electron flux data']

Information about ERG LEPe

PI:  ['Shiang-Yu Wang']
Affiliation:  ['Academia Sinica, Taiwan']

RoR of ERG project common: https://ergsc.isee.nagoya-u.ac.jp/data_info/rules_of_the_road.shtml.en
RoR of LEPe L2: https://ergsc.isee.nagoya-u.ac.jp/mw/index.php/ErgSat/Lepe

Contact: erg_lepe_info at isee.nagoya-u.ac.jp
**************************************************************************


26-Jan-26 11:17:54: Local file found: /mnt/j/observation_data//ergsc/satellite/erg/lepi/l2/3dflux/2022/09/erg_lepi_l2_3dflux_20220901_v03_00.cdf
26-Jan-26 11:17:58: Variable FPDU_Quality DEPEND_2 attribute FIDU_Channel has length 8, but corresponding data dimension has length 15. Removing attribute.
26-Jan-26 11:17:59: Variable FHEDU_Quality DEPEND_2 attribute FIDU_Channel has length 8, but corresponding data dimension has length 15. Removing attribute.
26-Jan-26 11:18:00: Variable FODU_Quality DEPEND_2 attribute FIDU_Channel has length 8, but corresponding data dimension has length 15. Removing attribute.
26-Jan-26 11:18:01: store_data: Data array for variable erg_lepi_l2_3dflux_FPDU_Quality has 4 dimensions, but only 1 v_n keys plus time. Adding empty v_n keys.
26-Jan-26 11:18:01: store_data: Data array for variable erg_lepi_l2_3dflux_FHEDU_Quality has 4 dimensions, but only 1 v_n keys plus time. Adding empty v_n keys.
26-Jan-26 11:18:02: store_data: Data array for variable erg_lepi_

 
**************************************************************************
['Exploration of Energization and Radiation in Geospace (ERG) Low Energy Particle Ion (LEPi) Experiment 3D ion flux data']

Information about ERG LEPi

PI:  ['Kazushi Asamura']
Affiliation:  ['ISAS, Jaxa']

RoR of ERG project common: https://ergsc.isee.nagoya-u.ac.jp/data_info/rules_of_the_road.shtml.en
RoR of LEPi L2: https://ergsc.isee.nagoya-u.ac.jp/mw/index.php/ErgSat/Lepi
RoR of ERG/LEPi: https://ergsc.isee.nagoya-u.ac.jp/mw/index.php/ErgSat/Lepi#Rules_of_the_Road

Contact: erg_lepi_info at isee.nagoya-u.ac.jp
**************************************************************************


26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FODU copied to erg_lepi_l2_3dflux_FODU_raw
26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FODU_sub copied to erg_lepi_l2_3dflux_FODU_sub_raw
26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FPEDU copied to erg_lepi_l2_3dflux_FPEDU_raw
26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FPEDU_sub copied to erg_lepi_l2_3dflux_FPEDU_sub_raw
26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FHEEDU copied to erg_lepi_l2_3dflux_FHEEDU_raw
26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FHEEDU_sub copied to erg_lepi_l2_3dflux_FHEEDU_sub_raw
26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FOEDU copied to erg_lepi_l2_3dflux_FOEDU_raw
26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FOEDU_sub copied to erg_lepi_l2_3dflux_FOEDU_sub_raw
26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FPDU_COUNT_RAW copied to erg_lepi_l2_3dflux_FPDU_COUNT_RAW_raw
26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FPDU_COUNT_RAW_sub copied to erg_lepi_l2_3dflux_FPDU_COUNT_RAW_sub_raw
26-Jan-26 11:18:03: erg_lepi_l2_3dflux_FHEDU_COUNT_RAW copied to erg_lepi_l2_3dflux_

 
 
**************************************************************************
['Exploration of Energization and Radiation in Geospace (ERG) Plasma Wave Experiment (PWE) High Frequency Analyzer (HFA) Level 3 UHR frequency and electron density data']

Information about ERG PWE HFA

PI:  ['Yoshiya Kasahara and Ayako Matsuoka']
Affiliation:  ['Kanazawa University and Kyoto University']

RoR of ERG project common: https://ergsc.isee.nagoya-u.ac.jp/data_info/rules_of_the_road.shtml.en
RoR of PWE/HFA: https://ergsc.isee.nagoya-u.ac.jp/mw/index.php/ErgSat/Pwe/Hfa

Contact: erg_pwe_info at isee.nagoya-u.ac.jp
**************************************************************************


26-Jan-26 11:18:07: erg_mgf_l2_mag_64hz_background_dsi copied to erg_mgf_l2_mag_64hz_background_dsi_pgs_temp
26-Jan-26 11:18:08: tinterpol (linear) was applied to: erg_mgf_l2_mag_64hz_background_dsi_pgs_temp
26-Jan-26 11:18:11: wildcard_expand: No match found for erg_lepe_l2_3dflux_FEDU_eflux
26-Jan-26 11:18:11: wildcard_expand: No match found for erg_lepe_l2_3dflux_FEDU_eflux
26-Jan-26 11:18:11: erg_mgf_l2_mag_64hz_background_dsi copied to erg_mgf_l2_mag_64hz_background_dsi_pgs_temp
26-Jan-26 11:18:11: tinterpol (linear) was applied to: erg_mgf_l2_mag_64hz_background_dsi_pgs_temp
26-Jan-26 11:18:11: /home/satanka/Documents/observation_workspace/.venv_pyspedas/lib/python3.12/site-packages/pyspedas/particles/moments/moments_3d.py:122: RuntimeWarning: invalid value encountered in divide
  velocity = flux/density/1e5 # km/s

26-Jan-26 11:18:16: erg_lepi_l2_3dflux_FPDU is 40% done.
26-Jan-26 11:18:21: erg_lepi_l2_3dflux_FPDU is 83% done.
26-Jan-26 11:18:25: wildcard_expand: No match found 

['erg_lepi_l2_3dflux_FODU_density',
 'erg_lepi_l2_3dflux_FODU_flux',
 'erg_lepi_l2_3dflux_FODU_mftens',
 'erg_lepi_l2_3dflux_FODU_velocity',
 'erg_lepi_l2_3dflux_FODU_ptens',
 'erg_lepi_l2_3dflux_FODU_ttens',
 'erg_lepi_l2_3dflux_FODU_vthermal',
 'erg_lepi_l2_3dflux_FODU_avgtemp']

In [36]:
ND_electron_LEP = pt.data_quants['erg_lepe_l2_3dflux_FEDU_density']
ND_electron_HFA = pt.data_quants['erg_pwe_hfa_l3_1min_ne_mgf']
Temp_electron   = pt.data_quants['erg_lepe_l2_3dflux_FEDU_avgtemp']

ND_proton       = pt.data_quants['erg_lepi_l2_3dflux_FPDU_density'].fillna(0)   # [/cc]
Flux_proton     = pt.data_quants['erg_lepi_l2_3dflux_FPDU_flux'].fillna(0)      # [/s/cm2]
Temp_proton     = pt.data_quants['erg_lepi_l2_3dflux_FPDU_avgtemp'].fillna(0)   # [eV]

ND_Helium       = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_density'].fillna(0)
Flux_Helium     = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_flux'].fillna(0)
Temp_Helium     = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_avgtemp'].fillna(0)

ND_Oxygen       = pt.data_quants['erg_lepi_l2_3dflux_FODU_density'].fillna(0)
Flux_Oxygen     = pt.data_quants['erg_lepi_l2_3dflux_FODU_flux'].fillna(0)
Temp_Oxygen     = pt.data_quants['erg_lepi_l2_3dflux_FODU_avgtemp'].fillna(0)

ND_ion          = ND_proton + ND_Helium + ND_Oxygen                                     # [/cc]
Flux_ion        = Flux_proton + Flux_Helium + Flux_Oxygen                               # [/s/cm2]
Ptot_ion        = ND_proton*Temp_proton + ND_Helium*Temp_Helium + ND_Oxygen*Temp_Oxygen # [eV/cc]

v_ion_dsi       = Flux_ion / ND_ion * 1E-2  # [m/s]
Temp_ion        = Ptot_ion / ND_ion         # [eV]

proton_mass_kg  = 1.6726219e-27  # kg
Helium_mass_kg  = proton_mass_kg * 4
Oxygen_mass_kg  = proton_mass_kg * 16

ion_mass        = (ND_proton*proton_mass_kg + ND_Helium*Helium_mass_kg + ND_Oxygen*Oxygen_mass_kg) / ND_ion #[kg]

In [37]:
R_interp    = R_DSI_to_FAC.interp(time=v_ion_dsi.time)

v_ion_fac   = xr.dot(v_ion_dsi, R_interp, dims='v_dim')
v_ion_fac   = v_ion_fac.dropna(dim='time', how='any')

print(v_ion_fac.time)
print(v_sc_fac.time)

26-Jan-26 11:18:47: /tmp/ipykernel_2396/3697766897.py:3: PendingDeprecationWarning: The `dims` argument has been renamed to `dim`, and will be removed in the future. This renaming is taking place throughout xarray over the next few releases.
  v_ion_fac   = xr.dot(v_ion_dsi, R_interp, dims='v_dim')



<xarray.DataArray 'time' (time: 1335)> Size: 11kB
array(['2022-09-01T21:00:52.591000064', '2022-09-01T21:01:00.591000064',
       '2022-09-01T21:01:08.607000064', ..., '2022-09-01T23:58:42.182000128',
       '2022-09-01T23:58:50.196999936', '2022-09-01T23:58:58.214000128'],
      shape=(1335,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 11kB 2022-09-01T21:00:52.591000064 ... 202...
<xarray.DataArray 'time' (time: 1782)> Size: 14kB
array(['2022-09-01T21:00:54.000000000', '2022-09-01T21:01:00.000000000',
       '2022-09-01T21:01:06.000000000', ..., '2022-09-01T23:58:48.000000000',
       '2022-09-01T23:58:54.000000000', '2022-09-01T23:59:00.000000000'],
      shape=(1782,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 14kB 2022-09-01T21:00:54 ... 2022-09-01T23...


In [38]:
v_sys_fac   = v_ion_fac.interp(time=v_sc_fac.time, method='linear') - v_sc_fac
v_sys_fac   = v_sys_fac.dropna(dim='time', how='any')
print(v_sys_fac)

<xarray.DataArray (time: 1781, axis: 3)> Size: 43kB
array([[ -34379.37357092,  -24244.78400612,   14155.65885751],
       [ -26959.14169954,  -19642.01291931,  -16257.01812357],
       [ -37318.293775  ,  -26985.81424576,    7409.20070628],
       ...,
       [-126645.44331937,  105892.533575  ,   10317.68252835],
       [-116575.13837988,   -4091.47005182,    3077.45245069],
       [ -67825.79729391,  -47154.09416813,   -8925.86103116]],
      shape=(1781, 3))
Coordinates:
  * axis     (axis) <U5 60B 'x_FAC' 'y_FAC' 'z_FAC'
  * time     (time) datetime64[ns] 14kB 2022-09-01T21:00:54 ... 2022-09-01T23...
Attributes:
    plot_options:  {'xaxis_opt': {'axis_label': '', 'crosshair': 'X', 'x_axis...


In [39]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_ion_dsi_analysis  = v_ion_dsi.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_ion_dsi_analysis.time, v_ion_dsi_analysis.data[:, 2]*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (DSI)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (DSI)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (DSI)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_ion_dsi_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_dsi.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [40]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_ion_fac_analysis  = v_ion_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_ion_fac_analysis.time, np.sqrt(v_ion_fac_analysis.data[:, 0]**2E0 + v_ion_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{ion}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_ion_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [41]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/23:09:00', '20220901/23:12:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_v_ion_fac        = (v_ion_fac.time.data[1] - v_ion_fac.time.data[0]) / np.timedelta64(1, 's')
#v_ion_fac_mean      = v_ion_fac.rolling(time=int(100/dt_v_ion_fac), center=True).mean()
#v_ion_fac_analysis_mean     = v_ion_fac_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_fac_analysis_mean.time, v_ion_fac_analysis_mean.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_ion_fac_analysis_mean.time, v_ion_fac_analysis_mean.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_ion_fac_analysis_mean.time, v_ion_fac_analysis_mean.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_ion_fac_analysis_mean.time, np.sqrt(v_ion_fac_analysis_mean.data[:, 0]**2E0 + v_ion_fac_analysis_mean.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{ion}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_ion_fac_analysis_mean.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_fac_mean_event3.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [42]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sys_fac_analysis  = v_sys_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_sys_fac_analysis.time, np.sqrt(v_sys_fac_analysis.data[:, 0]**2E0 + v_sys_fac_analysis.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sys_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [43]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_v_sys_fac        = (v_sys_fac.time.data[1] - v_sys_fac.time.data[0]) / np.timedelta64(1, 's')
#v_sys_fac_mean      = v_sys_fac.rolling(time=int(100/dt_v_sys_fac), center=True).mean()
#v_sys_fac_analysis_mean     = v_sys_fac_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 0]*1E-3, lw=1, c='k')
#ax_1.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 1]*1E-3, lw=1, c='k')
#ax_2.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 2]*1E-3, lw=1, c='k')
#ax_3.plot(v_sys_fac_analysis_mean.time, np.sqrt(v_sys_fac_analysis_mean.data[:, 0]**2E0 + v_sys_fac_analysis_mean.data[:, 1]**2E0)*1E-3, lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sys_fac_analysis_mean.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac_mean.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [44]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ND_proton_analysis  = ND_proton.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Helium_analysis  = ND_Helium.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Oxygen_analysis  = ND_Oxygen.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_ion_analysis     = ND_ion.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ion_mass_analysis  = ion_mass.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ND_proton_analysis.time,  ND_proton_analysis.data,    lw=1, c='k')
#ax_1.plot(ND_Helium_analysis.time,  ND_Helium_analysis.data,    lw=1, c='k')
#ax_2.plot(ND_Oxygen_analysis.time,  ND_Oxygen_analysis.data,    lw=1, c='k')
#ax_3.plot(ND_ion_analysis.time,     ND_ion_analysis.data,       lw=1, c='k')
#ax_4.plot(ion_mass_analysis.time,   ion_mass_analysis.data/proton_mass_kg, lw=1, c='k')
#
#ax_0.set_ylabel(r'$\mathrm{H}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_1.set_ylabel(r'$\mathrm{He}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_2.set_ylabel(r'$\mathrm{O}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_3.set_ylabel(r'ion' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_4.set_ylabel(r'ion mass' + '\n' + r'[$m_{\mathrm{p}}$]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_0.set_ylim(ymax=2)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_1.set_yscale('log')
#ax_1.set_ylim(ymin=1E-4)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_2.set_yscale('log')
#ax_2.set_ylim(ymin=1E-4, ymax=5E-2)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_3.set_yscale('log')
#ax_3.set_ylim(ymax=2)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_4.set_ylim(ymin=1, ymax=2.5)
#
#ax_4.set_xlim(ion_mass_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'ion_composition_ND.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [45]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_ND               = (ND_proton.time.data[1] - ND_proton.time.data[0]) / np.timedelta64(1, 's')
#
#ND_proton_mean  = ND_proton.rolling(time=int(100/dt_ND), center=True).mean()
#ND_Helium_mean  = ND_Helium.rolling(time=int(100/dt_ND), center=True).mean()
#ND_Oxygen_mean  = ND_Oxygen.rolling(time=int(100/dt_ND), center=True).mean()
#ND_ion_mean     = ND_ion.rolling(time=int(100/dt_ND), center=True).mean()
#ion_mass_mean   = ion_mass.rolling(time=int(100/dt_ND), center=True).mean()
#
#ND_proton_analysis_mean  = ND_proton_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Helium_analysis_mean  = ND_Helium_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_Oxygen_analysis_mean  = ND_Oxygen_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ND_ion_analysis_mean     = ND_ion_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ion_mass_analysis_mean   = ion_mass_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ND_proton_analysis_mean.time,  ND_proton_analysis_mean.data,    lw=1, c='k')
#ax_1.plot(ND_Helium_analysis_mean.time,  ND_Helium_analysis_mean.data,    lw=1, c='k')
#ax_2.plot(ND_Oxygen_analysis_mean.time,  ND_Oxygen_analysis_mean.data,    lw=1, c='k')
#ax_3.plot(ND_ion_analysis_mean.time,     ND_ion_analysis_mean.data,       lw=1, c='k')
#ax_4.plot(ion_mass_analysis_mean.time,   ion_mass_analysis_mean.data/proton_mass_kg, lw=1, c='k')
#
#ax_0.set_ylabel(r'$\mathrm{H}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_1.set_ylabel(r'$\mathrm{He}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_2.set_ylabel(r'$\mathrm{O}^{+}$' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_3.set_ylabel(r'ion' + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_4.set_ylabel(r'ion mass' + '\n' + r'[$m_{\mathrm{p}}$]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_0.set_ylim(ymax=2)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_1.set_yscale('log')
#ax_1.set_ylim(ymin=1E-4)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_2.set_yscale('log')
#ax_2.set_ylim(ymin=1E-4, ymax=5E-2)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_3.set_yscale('log')
#ax_3.set_ylim(ymax=2)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_4.set_ylim(ymin=1, ymax=2.5)
#
#ax_4.set_xlim(ion_mass_analysis_mean.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'ion_composition_ND_mean.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

- Alfvén speed
```math
v_{\mathrm{A}} := \frac{B_{0}}{\sqrt{\mu_{0} n_{\mathrm{e}} m_{\mathrm{i}}}}
```
- Ion thermal speed
```math
v_{\mathrm{thi}} := \sqrt{\frac{2 T_{\mathrm{i}}}{m_{\mathrm{i}}}}
```
- Ion acoustic speed
```math
c_{\mathrm{s}} := \sqrt{\frac{T_{\mathrm{e}}}{m_{\mathrm{i}}}}
```
- Proton cyclotron frequency
```math
f_{\mathrm{p}} := \frac{1}{2 \pi} \frac{e B_{0}}{m_{\mathrm{p}}}
```
- Ion plasma beta
```math
\beta_{\mathrm{i}} := \frac{2 \mu_{0} n_{\mathrm{e}} T_{\mathrm{i}}}{B_{0}^{2}} = \left( \frac{v_{\mathrm{thi}}}{v_{\mathrm{A}}} \right)^{2}
```
- Ion-to-electron temperature ratio
```math
\tau := \frac{T_{\mathrm{i}}}{T_{\mathrm{e}}} = \frac{1}{2} \left( \frac{v_{\mathrm{thi}}}{c_{\mathrm{s}}} \right)^{2}
```

In [46]:
da_B64_dsi_seg0_clean       = ds_B64_dsi_seg0_clean.to_dataarray(dim='v_dim').assign_coords(v_dim=np.arange(3))

print(da_B64_dsi_seg0_clean)

da_B64_dsi_seg0_clean_total = np.sqrt((da_B64_dsi_seg0_clean * da_B64_dsi_seg0_clean).sum(dim='v_dim'))

print(da_B64_dsi_seg0_clean_total)

<xarray.DataArray (v_dim: 3, time: 691238)> Size: 17MB
array([[1228.97625405, 1214.21877693, 1199.2404036 , ...,  161.50282937,
         161.37210507,  161.4583528 ],
       [ 280.45063963,  276.88621602,  273.59969068, ...,  -21.82199868,
         -21.86328607,  -21.94211938],
       [ 140.12046715,  138.23390709,  136.55342634, ..., -148.24320427,
        -148.21461142, -148.15601026]], shape=(3, 691238))
Coordinates:
  * time     (time) datetime64[ns] 6MB 2022-09-01T21:00:00.001522176 ... 2022...
  * v_dim    (v_dim) int64 24B 0 1 2
<xarray.DataArray (time: 691238)> Size: 6MB
array([1268.33313431, 1253.03704173, 1237.61107566, ...,  220.30753762,
        220.19657266,  220.22819908], shape=(691238,))
Coordinates:
  * time     (time) datetime64[ns] 6MB 2022-09-01T21:00:00.001522176 ... 2022...


In [47]:
B_total = da_B64_dsi_seg0_clean_total

time_base = B_total.time
print(time_base)

ND_electron_LEP_interp  = ND_electron_LEP.interp(time=time_base, method='linear')
ND_electron_HFA_interp  = ND_electron_HFA.interp(time=time_base, method='linear')
ND_electron_MID_interp  = (ND_electron_HFA_interp + ND_electron_LEP_interp) / 2E0


Temp_electron_interp    = Temp_electron.interp(time=time_base, method='linear')
Temp_ion_interp         = Temp_ion.interp(time=time_base, method='linear')
v_sys_fac_interp        = v_sys_fac.interp(time=time_base, method='linear')
ion_mass_interp         = ion_mass.interp(time=time_base, method='linear')

v_sys_fac_perp_interp   = xr.DataArray(
    data=np.sqrt(v_sys_fac_interp.data[:, 0]**2E0 + v_sys_fac_interp.data[:, 1]**2E0),
    dims=['time'],
    coords={'time': v_sys_fac_interp.time},
    attrs=v_sys_fac_interp.attrs
)

elementary_charge = 1.60218e-19  # C
mu0 = 4*np.pi*1e-7

Alfven_speed_LEP    = B_total*1E-9 / np.sqrt(mu0 * ND_electron_LEP_interp*1E6 * ion_mass_interp)
Alfven_speed_HFA    = B_total*1E-9 / np.sqrt(mu0 * ND_electron_HFA_interp*1E6 * ion_mass_interp)
Alfven_speed_MID    = B_total*1E-9 / np.sqrt(mu0 * ND_electron_MID_interp*1E6 * ion_mass_interp)

ion_thermal_speed   = np.sqrt(2E0 * Temp_ion_interp*elementary_charge / ion_mass_interp)
ion_acoustic_speed  = np.sqrt(Temp_electron_interp*elementary_charge / ion_mass_interp)

electron_mass_kg        = 9.1093837E-31
electron_thermal_speed  = np.sqrt(2E0 * Temp_electron_interp*elementary_charge / electron_mass_kg)

proton_cycl_freq    = elementary_charge * B_total*1E-9 / proton_mass_kg / 2E0 / np.pi

ion_plasma_beta_LEP = (ion_thermal_speed / Alfven_speed_LEP)**2E0
ion_plasma_beta_HFA = (ion_thermal_speed / Alfven_speed_HFA)**2E0
ion_plasma_beta_MID = (ion_thermal_speed / Alfven_speed_MID)**2E0

ion_to_electron_temp_ratio  = (ion_thermal_speed / ion_acoustic_speed)**2E0 / 2E0

# moving mean
dt_time_base            = (time_base.data[1] - time_base.data[0]) / np.timedelta64(1, 's')

Alfven_speed_LEP_mean   = Alfven_speed_LEP.rolling(time=int(100/dt_time_base), center=True).mean()
Alfven_speed_HFA_mean   = Alfven_speed_HFA.rolling(time=int(100/dt_time_base), center=True).mean()
Alfven_speed_MID_mean   = Alfven_speed_MID.rolling(time=int(100/dt_time_base), center=True).mean()

ion_thermal_speed_mean  = ion_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
electron_thermal_speed_mean = electron_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
ion_acoustic_speed_mean = ion_acoustic_speed.rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_perp_mean     = v_sys_fac_perp_interp.rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_x_mean        = v_sys_fac_interp[:, 0].rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_y_mean        = v_sys_fac_interp[:, 1].rolling(time=int(100/dt_time_base), center=True).mean()

ion_plasma_beta_LEP_mean            = ion_plasma_beta_LEP.rolling(time=int(100/dt_time_base), center=True).mean()
ion_plasma_beta_HFA_mean            = ion_plasma_beta_HFA.rolling(time=int(100/dt_time_base), center=True).mean()
ion_plasma_beta_MID_mean            = ion_plasma_beta_MID.rolling(time=int(100/dt_time_base), center=True).mean()

ion_to_electron_temp_ratio_mean = ion_to_electron_temp_ratio.rolling(time=int(100/dt_time_base), center=True).mean()
proton_cycl_freq_mean           = proton_cycl_freq.rolling(time=int(100/dt_time_base), center=True).mean()

# DataSet格納
ds_velocity_ms_perp = xr.Dataset(
    {
        'Alfven_speed_LEP':         Alfven_speed_LEP_mean,
        'Alfven_speed_MID':         Alfven_speed_MID_mean,
        'Alfven_speed_HFA':         Alfven_speed_HFA_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_sys_speed':           v_sys_fac_perp_mean
    }
)
ds_velocity_ms_perp = ds_velocity_ms_perp.dropna(dim='time', how='any')

print(ds_velocity_ms_perp)

ds_velocity_ms_toroidal = xr.Dataset(
    {
        'Alfven_speed_LEP':         Alfven_speed_LEP_mean,
        'Alfven_speed_MID':         Alfven_speed_MID_mean,
        'Alfven_speed_HFA':         Alfven_speed_HFA_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_sys_speed':           v_sys_fac_x_mean
    }
)
ds_velocity_ms_toroidal = ds_velocity_ms_toroidal.dropna(dim='time', how='any')

print(ds_velocity_ms_toroidal)

ds_velocity_ms_poloidal = xr.Dataset(
    {
        'Alfven_speed_LEP':         Alfven_speed_LEP_mean,
        'Alfven_speed_MID':         Alfven_speed_MID_mean,
        'Alfven_speed_HFA':         Alfven_speed_HFA_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_sys_speed':           v_sys_fac_y_mean
    }
)
ds_velocity_ms_poloidal = ds_velocity_ms_poloidal.dropna(dim='time', how='any')

print(ds_velocity_ms_poloidal)

ds_parameter = xr.Dataset(
    {
        'ion_plasma_beta_LEP':      ion_plasma_beta_LEP_mean,
        'ion_plasma_beta_MID':      ion_plasma_beta_MID_mean,
        'ion_plasma_beta_HFA':      ion_plasma_beta_HFA_mean,
        'i-e_temp_ratio':           ion_to_electron_temp_ratio_mean,
        'proton_cycl_freq_Hz':      proton_cycl_freq_mean,
        'number_density_LEP_cc':    ND_electron_LEP_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'number_density_MID_cc':    ND_electron_MID_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'number_density_HFA_cc':    ND_electron_HFA_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'ion_mass_kg':              ion_mass_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_ion_eV':              Temp_ion_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_electron_eV':         Temp_electron_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'B_total_nT':               B_total.rolling(time=int(100/dt_time_base), center=True).mean()
    }
)
ds_parameter = ds_parameter.dropna(dim='time', how='any')

print(ds_parameter)

<xarray.DataArray 'time' (time: 691238)> Size: 6MB
array(['2022-09-01T21:00:00.001522176', '2022-09-01T21:00:00.017222016',
       '2022-09-01T21:00:00.032822016', ..., '2022-09-01T23:59:54.219361920',
       '2022-09-01T23:59:54.234962176', '2022-09-01T23:59:54.250561920'],
      shape=(691238,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 6MB 2022-09-01T21:00:00.001522176 ... 2022...
<xarray.Dataset> Size: 43MB
Dimensions:                 (time: 677553)
Coordinates:
  * time                    (time) datetime64[ns] 5MB 2022-09-01T21:01:43.722...
Data variables:
    Alfven_speed_LEP        (time) float64 5MB 1.77e+08 1.77e+08 ... 3.977e+07
    Alfven_speed_MID        (time) float64 5MB 3.771e+06 3.771e+06 ... 1.498e+07
    Alfven_speed_HFA        (time) float64 5MB 2.667e+06 2.667e+06 ... 1.099e+07
    ion_thermal_speed       (time) float64 5MB 3.387e+04 3.387e+04 ... 5.154e+05
    electron_thermal_speed  (time) float64 5MB 2.638e+07 2.638e+07 ... 7.096e+06

In [48]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       lw=1, c='b')
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       lw=1, c='k')
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       lw=1, c='r')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#ax_4.set_xlim(ds_velocity_ms_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_perp.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [49]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       lw=1, c='b')
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       lw=1, c='k')
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       lw=1, c='r')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}x}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#ax_4.set_xlim(ds_velocity_ms_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_toroidal.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [50]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_LEP']*1E-3,       lw=1, c='b')
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,       lw=1, c='k')
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_HFA']*1E-3,       lw=1, c='r')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,      lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3,     lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,         lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}y}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_0.set_yscale('log')
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#ax_4.set_xlim(ds_velocity_ms_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_poloidal.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [53]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_velocity_ms_poloidal_analysis    = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
ds_velocity_ms_toroidal_analysis    = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
from datetime import datetime
import matplotlib.dates as mdates

mpl.rcParams['font.size'] = 25

fig = plt.figure(figsize=(11, 21))
gs = fig.add_gridspec(7, 1)
ax_0 = fig.add_subplot(gs[0, 0])
ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
#ax_7 = fig.add_subplot(gs[7, 0], sharex=ax_0)

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)
ax_4.tick_params(axis='x', which='both', labelbottom=False)
ax_5.tick_params(axis='x', which='both', labelbottom=False)
#ax_6.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_MID_cc'],       lw=1, c='k')
ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],                 lw=1, c='red', label=r'$T_{\mathrm{i}}$')
ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],            lw=1, c='blue', label=r'$T_{\mathrm{e}}$')
ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],                  lw=1, c='k')
ax_3.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_mass_kg']/proton_mass_kg,  lw=1, c='k')
ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta_MID'],         lw=1, c='k', label=r'$\beta_{\mathrm{i}}$')
ax_4.plot(ds_parameter_analysis.time, electron_mass_kg / ds_parameter_analysis['ion_mass_kg'], lw=2, c='green', linestyle='-.', label=r'$m_{\mathrm{e}}/m_{\mathrm{i}}$')
ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=1, c='b', label=r'$v_{\mathrm{the}}$')
ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed_MID']*1E-3,   lw=1, c='k', label=r'$v_{\mathrm{A}}$')
ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='red')
#ax_7.plot(ds_velocity_ms_toroidal_analysis.time, ds_velocity_ms_toroidal_analysis['perp_sys_speed']*1E-3, lw=1, c='orange', label=r'$v_{\mathrm{sys}x}$')
#ax_7.plot(ds_velocity_ms_poloidal_analysis.time, ds_velocity_ms_poloidal_analysis['perp_sys_speed']*1E-3, lw=1, c='green', label=r'$v_{\mathrm{sys}y}$')

ax_0.set_ylabel(r'$n_{\mathrm{e}}$'                     + '\n' + r'[$\mathrm{cm}^{-3}$]')
ax_1.set_ylabel(r'$T_{\mathrm{i}}$, $T_{\mathrm{e}}$'   + '\n' + '[eV]')
ax_2.set_ylabel(r'$B_{0}$'                              + '\n' + '[nT]')
ax_3.set_ylabel(r'$m_{\mathrm{i}}$'                     + '\n' + r'[$m_{\mathrm{H}}$]')
ax_4.set_ylabel(r'$\beta_{\mathrm{i}}$')
ax_5.set_ylabel(r'$v_{\mathrm{the}}$, $v_{\mathrm{A}}$' + '\n' + '[km/s]')
ax_6.set_ylabel(r'$v_{\mathrm{thi}}$'                   + '\n' + '[km/s]')
#ax_7.set_ylabel(r'$v_{\mathrm{sys}\perp}$'              + '\n' + '[km/s]')

ax_1.set_ylim(ymin=0)
ax_3.set_ylim(ymin=1, ymax=2)
ax_4.set_yscale('log')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.5)
ax_3.minorticks_on()
ax_3.grid(which='both', alpha=0.5)
ax_4.minorticks_on()
ax_4.grid(which='both', alpha=0.5)
ax_5.minorticks_on()
ax_5.grid(which='both', alpha=0.5)
ax_6.minorticks_on()
ax_6.grid(which='both', alpha=0.5)
#ax_7.minorticks_on()
#ax_7.grid(which='both', alpha=0.5)

ax_1.legend(fontsize=20, ncol=2)
ax_4.legend(fontsize=20, ncol=2)
ax_5.legend(fontsize=20, ncol=2)
#ax_7.legend(fontsize=20, ncol=2)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_6.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax_6.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax_6.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

def add_panel_label(ax, label, x=-0.15, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

def to_py_datetime(t_np64):
    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)

# 軌道データ
pos_da = pt.data_quants['erg_orb_l2_pos_rmlatmlt']  # (Nt, 3)
t_pos_py = to_py_datetime(pos_da.time.values)
t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数

R   = np.asarray(pos_da.values[:, 0], dtype=float)  # Re
mlat= np.asarray(pos_da.values[:, 1], dtype=float)  # deg
mlt = np.asarray(pos_da.values[:, 2], dtype=float)  # hour [0,24)

L_shell = R / np.cos(np.deg2rad(mlat))**2E0

# --- MLT の 24h 周期をほどいてから補間し、最後に 24 で折り返す ---
mlt_unwrap = np.unwrap(mlt * 2*np.pi/24.0) * 24.0/(2*np.pi)

# 補間関数（tick の x は「日数」なのでそのまま使う）
def interp_at(x_num):
    #Ri    = np.interp(x_num, t_pos_num, R, left=np.nan, right=np.nan)
    Ri    = np.interp(x_num, t_pos_num, L_shell, left=np.nan, right=np.nan)     # L-shellを示す
    mlati = np.interp(x_num, t_pos_num, mlat, left=np.nan, right=np.nan)
    mltiu = np.interp(x_num, t_pos_num, mlt_unwrap, left=np.nan, right=np.nan)
    mlti  = np.mod(mltiu, 24.0)
    return Ri, mlati, mlti

# 目盛フォーマッタ
def rmlt_formatter(x, pos=None):
    Ri, mlati, mlti = interp_at(x)
    if np.any(~np.isfinite([Ri, mlati, mlti])):
        return ""  # 範囲外は空
    return (f"{Ri:0.2f}\n"
            f"{mlati:0.2f}\n"
            f"{mlti:0.2f}")

# セカンダリ x 軸（底 side）を作ってラベルを差し替え
secax = ax_6.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
secax.xaxis.set_major_formatter(mticker.FuncFormatter(rmlt_formatter))

# メインの時間ラベルと重ならないよう余白を広げる
ax_6.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル

# 好みで：目盛間隔をメイン x と合わせる
secax.set_ticks(ax_6.get_xticks())

fig.text(0.07, 0.067, "hhmm", ha='center', va='center')
#fig.text(0.07, 0.047, r"R [$R_{\mathrm{E}}$]", ha='center', va='center')
fig.text(0.07, 0.047, r"L-shell", ha='center', va='center')
fig.text(0.07, 0.027, r"MLAT", ha='center', va='center')
fig.text(0.07, 0.007, r"MLT", ha='center', va='center')

add_panel_label(ax_0, '(l)')
add_panel_label(ax_1, '(m)')
add_panel_label(ax_2, '(n)')
add_panel_label(ax_3, '(o)')
add_panel_label(ax_4, '(p)')
add_panel_label(ax_5, '(q)')
add_panel_label(ax_6, '(r)')
#add_panel_label(ax_7, '(t)')

fig.suptitle('Arase', y=0.99)

fig.subplots_adjust(hspace=0)
fig.tight_layout(pad=0)

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'Figure_1_d.png')
    print(fig_path)
    fig.savefig(fig_path)
    fig.savefig(os.path.join(path_base_save_plot, 'Figure_1_d.pdf'))
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330/Figure_1_d.png


# KAWの確認に適した時間窓$T_{\mathrm{window}}$の検討

```math
\frac{1}{v_{\mathrm{A}}} \frac{|\bf{E}_{\perp}|}{|\bf{B}_{\perp}|} = \frac{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2}}{\sqrt{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2} \left( 1 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)}} = \sqrt{10} \\
k_{\perp} \rho_{\mathrm{i}} = \frac{f_{\mathrm{sc}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \\
\therefore T_{\mathrm{window}} := \frac{1}{f_{\mathrm{sc}}} = \frac{1}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \left[ 9 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \left\{ 10 + \sqrt{117 \left( \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)^{2} + 180 \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} + 100} \right\} \right]^{-\frac{1}{2}}
```

In [ ]:
def dedup_and_sort(ds):
    ds = ds.sortby("time")
    t = ds["time"].values
    _, keep = np.unique(t, return_index=True)  # 先勝ちで一意化
    return ds.isel(time=np.sort(keep))

ds_parameter_clean = dedup_and_sort(ds_parameter)
ds_velocity_ms_perp_clean = dedup_and_sort(ds_velocity_ms_perp)
ds_velocity_ms_poloidal_clean = dedup_and_sort(ds_velocity_ms_poloidal)
ds_velocity_ms_toroidal_clean = dedup_and_sort(ds_velocity_ms_toroidal)

ds_parameter_interp = ds_parameter_clean.interp(time=ds_velocity_ms_perp_clean.time)
print(ds_parameter_interp)
print(ds_velocity_ms_perp_clean)

In [ ]:
T_window = ds_parameter_interp['ion_mass_kg'] / proton_mass_kg / ds_parameter_interp['proton_cycl_freq_Hz'].data * ds_velocity_ms_perp_clean['ion_thermal_speed'].data / ds_velocity_ms_perp_clean['perp_sys_speed'].data / np.sqrt(9. + 1. / ds_parameter_interp['i-e_temp_ratio'].data * (10. + np.sqrt(117. / (ds_parameter_interp['i-e_temp_ratio'].data)**(2.) + 180. / ds_parameter_interp['i-e_temp_ratio'].data + 100.)))

da_T_window = xr.DataArray(data=T_window, dims=('time'), coords={'time': ds_velocity_ms_perp_clean.time}, name='T_window')
da_T_window

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_T_window_analysis = da_T_window.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

print(np.nanmin(da_T_window_analysis))

In [ ]:
#import matplotlib as mpl
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#da_T_window_analysis = da_T_window.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#mpl.rcParams['font.size'] = 15
#fig = plt.figure(figsize=(10, 4))
#ax = fig.add_subplot(111)
#ax.plot(da_T_window_analysis.time, da_T_window_analysis.data, c='k', lw=1)
#ax.minorticks_on()
#ax.set_ylabel(r'$T_{\mathrm{window}}$' + '\n[sec]')
#ax.set_yscale('log')
#ax.set_ylim(ymin=0.1)
#ax.grid(which='both', alpha=0.5)
#ax.set_xlim(da_T_window_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#print(np.nanmin(da_T_window_analysis), np.nanmean(da_T_window_analysis))
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'T_window.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# KAWが期待される周波数の最小値$f_{\mathrm{predict}}$の検討

```math
k_{\perp} \rho_{\mathrm{i}} = \frac{f_{\mathrm{sc}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \\
f_{\mathrm{predict}} := 5 f_{\mathrm{ci}} \frac{V_{\mathrm{sys}\perp}}{v_{\mathrm{thi}}}
```

In [ ]:
da_f_predict    = 5E0 * ds_parameter_clean['proton_cycl_freq_Hz'] * ds_velocity_ms_perp_clean['perp_sys_speed'] / ds_velocity_ms_perp_clean['ion_thermal_speed']

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_f_predict_window     = da_f_predict.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

print(np.nanmin(da_f_predict_window))

In [ ]:
#import matplotlib as mpl
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#da_f_predict_window     = da_f_predict.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#mpl.rcParams['font.size'] = 15
#fig = plt.figure(figsize=(10, 4))
#ax = fig.add_subplot(111)
#ax.plot(da_f_predict_window.time, da_f_predict_window.data, c='k', lw=1)
#ax.minorticks_on()
#ax.set_ylabel(r'$f_{\mathrm{predict}}$' + '\n[Hz]')
#ax.grid(which='both', alpha=0.5)
#ax.set_xlim(da_T_window_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#print(np.nanmin(da_T_window_analysis), np.nanmean(da_T_window_analysis))
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'f_predict.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# Wavelet analysis

In [ ]:
import os
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt
import pywt

import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.tdwavelet_themis as tw
import importlib
importlib.reload(tw)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm

def generate_red_noise(N, g=0.72):
    """AR(1)モデルによるレッドノイズの生成"""
    noise = np.random.randn(N)
    red_noise = np.zeros(N)
    for i in range(1, N):
        red_noise[i] = g * red_noise[i-1] + noise[i]
    return red_noise

def run_wco_monte_carlo(fs, n_iterations=300, n_points=2000, g=0.72):
    """
    指定されたfsに対して95%有意水準を計算する
    """
    dt = 1.0 / fs
    s0 = 2.0
    dj = 1.0 / 32.0
    
    # 解析用のJをデータ長に合わせて計算 (簡易版)
    J = int(np.ceil(np.log2((n_points * dt / 3.0) / s0) / dj))
    
    all_wco_values = []

    print(f"Starting Monte Carlo for fs={fs} Hz ({n_iterations} iterations)...")
    
    for _ in tqdm(range(n_iterations)):
        # 1. 無相関なレッドノイズペアの生成
        d1 = generate_red_noise(n_points, g=g)
        d2 = generate_red_noise(n_points, g=g)
        
        # 2. xarray Dataset化 (cwt_from_datasetが受け取れる形式)
        time = (np.arange(n_points) * dt * 1e9).astype('int64').astype('datetime64[ns]')
        ds_sim = xr.Dataset({
            "E": ("time", d1),
            "B": ("time", d2)
        }, coords={"time": time})
        
        # 3. CWT計算 (apply_coi_mask=Falseにして全点使う)
        ds_cwt = tw.cwt_from_dataset(
            ds_sim, dt=dt, s0=s0, dj=dj, J=J, 
            variables=["E", "B"], apply_coi_mask=False
        )
        
        # 4. WCO計算 (Ultra版/FFT版)
        wco, _ = tw.calculate_xwt_wco(ds_cwt, "E_coef", "B_coef", dt, dj)
        
        # 端点効果を避けるため中央部分を抽出して蓄積
        # (WCOの分布は基本スケールに依存しないはずなので全スケールまとめる)
        mid = n_points // 4
        all_wco_values.append(wco[mid:-mid, :].flatten())

    # 5. 95パーセンタイルを算出
    all_wco_values = np.concatenate(all_wco_values)
    sig95 = np.percentile(all_wco_values, 95)
    
    return sig95

# ---- 実行部分 ----
fs_targets = [64.0]
significance_level = {}

for fs in fs_targets:
    # 128Hzなどは点数が多いと重いので n_points を調整しても良い
    # 理論上、平滑化窓内の「独立な点数」が支配的なので、n_pointsは一定でも傾向は出る
    sig_level = run_wco_monte_carlo(fs, n_iterations=300, n_points=20000)
    significance_level[fs] = sig_level
    print(f"fs = {fs:>8} Hz -> 95% Significance Level: {sig_level:.4f}")

# 結果の可視化
plt.figure(figsize=(8, 5))
plt.bar([str(f) for f in fs_targets], [significance_level[f] for f in fs_targets], color='skyblue')
plt.ylabel("95% Significance Level (WCO)")
plt.xlabel("Sampling Frequency [Hz]")
plt.title("WCO Significance Level vs Sampling Frequency")
plt.grid(axis='y', linestyle='--')
plt.show()

In [ ]:
import os
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt
import pywt

fs = 64.0
dt = 1.0 / fs
s0 = 2.0
dj = 1.0/32.0

f_target = 0.01         # 目標最低周波数
N_cycle_min = 3.0       # 最低3周期は欲しい

# ---- 1) 全セグメントの長さを調べる ----
T_list = []
for ds_seg in ds_EB64_fac_segs:
    t0 = ds_seg.time.values[0]
    t1 = ds_seg.time.values[-1]
    T_seg = (t1 - t0) / np.timedelta64(1, 's')
    T_list.append(T_seg)

T_max = max(T_list)
print("max segment length [s] =", T_max)

# ---- 2) 最長セグメントで意味のある最低周波数 ----
f_min_phys = N_cycle_min / T_max
f_min_seg = max(f_target, f_min_phys)
print("f_min_seg (for longest segment) =", f_min_seg)

# ---- 3) その周波数に対応する scale_max ----
scale_target = 1.0 / (f_min_seg * dt)

# ---- 4) J_longest を計算 ----
J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
print("J_longest =", J_longest)

vars_64 = ['E64_fac_x','E64_fac_y','E64_fac_z', 'B64_fac_x','B64_fac_y','B64_fac_z']

ds_EB64_fac_cwt_segs    = []

for ds_seg in ds_EB64_fac_segs:
    ds_cwt = tw.cwt_from_dataset(
        ds_seg,
        dt=dt,
        s0=s0,
        dj=dj,
        J=J_longest,
        variables=vars_64
    )
    if ds_cwt.data_vars:
        ds_EB64_fac_cwt_segs.append(ds_cwt)

print(ds_EB64_fac_cwt_segs)

In [ ]:
ds_EB64_fac_xwt_wco_segs = []

def process_segment_EB64(ds_cwt, dt, dj):

    # XWT / WCO計算
    wco_exby, phase_exby = tw.calculate_xwt_wco(ds_cwt, 'E64_fac_x_coef', 'B64_fac_y_coef', dt, dj)
    wco_eybx, phase_eybx = tw.calculate_xwt_wco(ds_cwt, 'E64_fac_y_coef', 'B64_fac_x_coef', dt, dj)

    # 結果の格納
    ds_xwt = xr.Dataset({
        "EB64_wco_exby": (("time", "freq"), wco_exby.astype(np.float32)),
        "EB64_phase_exby": (("time", "freq"), phase_exby.astype(np.float32)),
        "EB64_wco_eybx": (("time", "freq"), wco_eybx.astype(np.float32)),
        "EB64_phase_eybx": (("time", "freq"), phase_eybx.astype(np.float32))
    }, coords=ds_cwt.coords)

    return ds_xwt

for ds_seg in ds_EB64_fac_cwt_segs:
    ds_xwt = process_segment_EB64(ds_seg, dt, dj)
    ds_EB64_fac_xwt_wco_segs.append(ds_xwt)

print(ds_EB64_fac_xwt_wco_segs)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

# ---- セグメント連結（freq合わせ）----
def concat_cwt_segments(dsets, var):
    # dsets をリストに正規化
    if isinstance(dsets, xr.Dataset):
        dsets = [dsets]
    elif isinstance(dsets, (str, bytes)):
        raise TypeError("dsets は Dataset のリストにして")

    das = []
    for ds in dsets:
        if ds is None or not isinstance(ds, xr.Dataset):
            continue
        if var in ds.data_vars:
            das.append(ds[var])

    if not das:
        return None, None

    pow_cat = xr.concat(das, dim="time").sortby("time")

    coi_name = var.replace("_cwt", "_coi")
    coi_list = []
    for ds in dsets:
        if isinstance(ds, xr.Dataset) and coi_name in ds.data_vars:
            coi_list.append(ds[coi_name])
    coi_cat = xr.concat(coi_list, dim="time").sortby("time") if coi_list else None
    return pow_cat, coi_cat

# ---- 1面描画：外でax/caxを用意する ----
def plot_cwt_on_ax(ax, da_pow, da_coi=None, t0=None, t1=None, minutes=5,
                   zrange=(1e-6, 1e3), yrange=(1e-2, 4.0),
                   cmap="turbo", label_left="", unit_right=""):
    # 時間切り出し
    if t0 is not None:
        if t1 is None:
            t1 = t0 + np.timedelta64(minutes, "m")
        da = da_pow.sel(time=slice(t0, t1))
        coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None else None
    else:
        da, coi = da_pow, da_coi
    if da.time.size == 0: return None, None

    T = mdates.date2num(da.time.values)
    F = da.freq.values
    Z = da.values.astype(float)

    # COIマスク（低周波側をNaN）
    if coi is not None:
        C = coi.values[:, None]
        Z = np.where(F[None, :] < C, np.nan, Z)

    # メッシュ
    Tm = np.tile(T, (F.size, 1)).T
    Fm = np.tile(F, (T.size, 1))

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto",
                        norm=LogNorm(vmin=zrange[0], vmax=zrange[1]), cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    if t0 is not None:
        if t1 is None:
            t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)


    # 右側カラーバー
    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(f"{unit_right}")
    return pcm, cb


In [ ]:
targets = [
    ("E64_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E64_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E64_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B64_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B64_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B64_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]
joined_EB64_fac_cwt = {}
for v, _, _ in targets:
    da_, coi = concat_cwt_segments(ds_EB64_fac_cwt_segs, v)
    da_ = da_.sortby('freq')
    if da_ is not None: joined_EB64_fac_cwt[v] = (da_.sortby("freq"), coi)

In [ ]:
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330/wavelet_PSD"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#time_windows = [
#    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
#    for n in range(36)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets):
#        if v not in joined_EB64_fac_cwt: continue
#        da, coi = joined_EB64_fac_cwt[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
def concat_xwt_segments(dsets, var):
    das = []
    for ds in dsets:
        if ds is not None and var in ds.data_vars:
            das.append(ds[var])
    if not das: return None
    return xr.concat(das, dim="time").sortby("time")

from matplotlib.colors import Normalize

def plot_xwt_phase_on_ax(ax, da_wco, da_phase, t0=None, minutes=5,
                         wco_thresh=0, yrange=(1e-2, 4.0),
                         mode="wco", label_left=""):
    # 時間切り出し
    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        wco = da_wco.sel(time=slice(t0, t1))
        phase = da_phase.sel(time=slice(t0, t1))
    else:
        wco, phase = da_wco, da_phase

    if wco.time.size == 0: return None

    T = mdates.date2num(wco.time.values)
    F = wco.freq.values
    Tm, Fm = np.meshgrid(T, F, indexing='ij')

    if mode == "wco":
        Z = wco.values.astype(float)
        cmap = "viridis"
        norm = Normalize(vmin=0, vmax=1)
        unit = "Coherency"
    else: # mode == "phase"
        # WCOが低い領域をマスクする
        Z = np.abs(phase.where(wco > wco_thresh).values.astype(float))
        cmap = "Spectral"
        norm = Normalize(vmin=0, vmax=180)
        unit = "Phase (abs) [deg]"

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto", norm=norm, cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)

    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(unit)
    if mode == "phase":
        cb.set_ticks([0, 45, 90, 135, 180])
    
    return pcm, cb

In [ ]:
time_windows = [
    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
    for n in range(36)
]

targets_EB64_xwt = [
    ("EB64_wco_exby", "EB64_phase_exby", r"$E_{x}-B_{y}$"),
    ("EB64_wco_eybx", "EB64_phase_eybx", r"$E_{y}-B_{x}$"),
]
joined_EB64_xwt = {}
for w_var, p_var, lab in targets_EB64_xwt:
    w_da = concat_xwt_segments(ds_EB64_fac_xwt_wco_segs, w_var)
    p_da = concat_xwt_segments(ds_EB64_fac_xwt_wco_segs, p_var)
    if w_da is not None:
        joined_EB64_xwt[lab] = (w_da, p_da)

#for t0 in time_windows:
#    fig, axes = plt.subplots(4, 1, figsize=(10, 12), sharex=True)
#    
#    for i, (w_var, p_var, lab) in enumerate(targets_EB64_xwt):
#        if lab not in joined_EB64_xwt: continue
#        w_da, p_da = joined_EB64_xwt[lab]
#        
#        # WCOプロット
#        plot_xwt_phase_on_ax(axes[2*i], w_da, p_da, t0=t0, mode="wco", wco_thresh=significance_level[64.0],
#                             label_left=f"Coherency ({lab})", yrange=(np.nanmin(p_da.freq), np.nanmax(p_da.freq)))
#        
#        # Phaseプロット
#        plot_xwt_phase_on_ax(axes[2*i+1], w_da, p_da, t0=t0, mode="phase", wco_thresh=significance_level[64.0],
#                             label_left=f"Phase ({lab})", yrange=(np.nanmin(p_da.freq), np.nanmax(p_da.freq)))
#
#    axes[-1].set_xlabel("time")
#    fig.suptitle('95% Significance Level (WCO) = ' + f'{significance_level[64.0]:.4f}')
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_64_xwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

# 5分毎のPSDのMedianをplot

In [ ]:
da_E64_fac_x_cwt    = joined_EB64_fac_cwt["E64_fac_x_cwt"]
da_E64_fac_y_cwt    = joined_EB64_fac_cwt["E64_fac_y_cwt"]
da_E64_fac_z_cwt    = joined_EB64_fac_cwt["E64_fac_z_cwt"]
da_B64_fac_x_cwt    = joined_EB64_fac_cwt["B64_fac_x_cwt"]
da_B64_fac_y_cwt    = joined_EB64_fac_cwt["B64_fac_y_cwt"]
da_B64_fac_z_cwt    = joined_EB64_fac_cwt["B64_fac_z_cwt"]

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 20

# ---- 入力 ----
pairs = [
    ("E_64_FAC_x_cwt",   da_E64_fac_x_cwt),
    ("E_64_FAC_y_cwt",   da_E64_fac_y_cwt),
    ("E_64_FAC_z_cwt",   da_E64_fac_z_cwt),
    ("B_64_FAC_x_cwt",   da_B64_fac_x_cwt),
    ("B_64_FAC_y_cwt",   da_B64_fac_y_cwt),
    ("B_64_FAC_z_cwt",   da_B64_fac_z_cwt),
]
t_all_start = np.datetime64('2022-09-01T21:00:00')
t_all_end   = np.datetime64('2022-09-02T00:00:00')
step        = np.timedelta64(5, 'm')  # 5分
outdir      = (
    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330/wavelet_PSD/5min_PSD"
)
os.makedirs(outdir, exist_ok=True)

# 事前に CWT 本体のみ取り出し、timeでソート
pairs_sorted = []
for name, da in pairs:
    if isinstance(da, tuple):
        da = da[0]
    if isinstance(da, xr.DataArray):
        pairs_sorted.append((name, da.sortby('time')))

def compute_median_dict(pairs_sorted, t0, t1):
    d = {}
    for name, da in pairs_sorted:
        sub = da.sel(time=slice(t0, t1))
        if sub.sizes.get('time', 0) == 0:
            continue
        if np.iscomplexobj(sub.data):
            sub = (sub.real**2 + sub.imag**2)
        d[name] = sub.median(dim='time', skipna=True)  # (freq,)
    return d

def plot_median_dict(mdict, t0, t1, outdir=None):
    fig, ax = plt.subplots(figsize=(8, 8))

    color_map = {
        ('E', 'x'): 'blue',      # Ex
        ('E', 'y'): 'orange',    # Ey
        ('E', 'z'): 'brown',     # Ez
        ('B', 'x'): 'green',     # Bx
        ('B', 'y'): 'red',       # By
        ('B', 'z'): 'purple',    # Bz
    }

    for name, med in mdict.items():
        if name.split('_')[3] == 'z':
            continue
        prefix  = name.split('_')[0]
        comp    = name.split('_')[3]
        coor    = name.split('_')[2]
        label  = f"${prefix}_{comp}$ ({coor})"
        col     = color_map.get((prefix, comp), 'gray')
        ax.loglog(med['freq'], med, label=label, lw=1, color=col)

    ax.minorticks_on()
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Median PSD')
    ax.set_title(f"Median {str(t0)[11:]}–{str(t1)[11:]}\n (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)")
    ax.grid(True, which='both', ls=':')
    ax.set_yticks([1E-8, 1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
    ax.legend(ncol=2, fontsize=15)
    ax.set_xlim(1e-2, 32)
    ax.set_ylim(1e-8, 1e4)
    plt.tight_layout()

    if outdir and os.path.isdir(outdir):
        fn = f"median_bs_E_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
        fig.savefig(os.path.join(outdir, fn), dpi=300, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

## ---- 5分窓でループ ----
#t_starts = np.arange(t_all_start, t_all_end, step)  # 21:00, 21:05, ..., 23:25
#for t0 in t_starts:
#    t1 = t0 + step
#    mdict = compute_median_dict(pairs_sorted, t0, t1)
#    if not mdict:  # その窓でデータ無し
#        continue
#    plot_median_dict(mdict, t0, t1, outdir=outdir)

t_starts    = np.datetime64('2022-09-01T23:09:00')
t_ends      = np.datetime64('2022-09-01T23:12:00')
mdict       = compute_median_dict(pairs_sorted, t_starts, t_ends)
plot_median_dict(mdict, t_starts, t_ends, outdir)

# 各成分のNoise (Median) を抽出

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

pairs_sorted = []

pairs = [
    ("E64_fac_x_cwt",   da_E64_fac_x_cwt),
    ("E64_fac_y_cwt",   da_E64_fac_y_cwt),
    ("E64_fac_z_cwt",   da_E64_fac_z_cwt),
    ("B64_fac_x_cwt",   da_B64_fac_x_cwt),
    ("B64_fac_y_cwt",   da_B64_fac_y_cwt),
    ("B64_fac_z_cwt",   da_B64_fac_z_cwt),
]

for name, da in pairs:
    if isinstance(da, tuple):
        da = da[0]
    if isinstance(da, xr.DataArray):
        pairs_sorted.append((name, da.sortby('time')))

noise_t0, noise_t1 = np.datetime64('2022-09-01T21:30:00'), np.datetime64('2022-09-01T21:55:00')

def compute_median_dict(pairs_sorted, t0, t1):
    d = {}
    for name, da in pairs_sorted:
        sub = da.sel(time=slice(t0, t1))
        if sub.sizes.get('time', 0) == 0:
            continue
        if np.iscomplexobj(sub.data):
            sub = (sub.real**2 + sub.imag**2)
        d[name] = sub.median(dim='time', skipna=True)  # (freq,)
    return d

noise_da_dict   = compute_median_dict(pairs_sorted, noise_t0, noise_t1)
ds_noise_median = xr.Dataset(noise_da_dict)

In [ ]:
## ---- 入力 ----
#pairs = [
#    ("E_64_FAC_x_cwt",   da_E64_fac_x_cwt),
#    ("E_64_FAC_y_cwt",   da_E64_fac_y_cwt),
#    ("E_64_FAC_z_cwt",   da_E64_fac_z_cwt),
#    ("B_64_FAC_x_cwt",   da_B64_fac_x_cwt),
#    ("B_64_FAC_y_cwt",   da_B64_fac_y_cwt),
#    ("B_64_FAC_z_cwt",   da_B64_fac_z_cwt),
#]
#outdir      = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330/wavelet_PSD"
#)
#os.makedirs(outdir, exist_ok=True)
#
#def plot_median_dict(mdict, t0, t1, outdir=None):
#    fig, ax = plt.subplots(figsize=(8, 8))
#    for name, med in mdict.items():
#        prefix  = name.split('_')[0][0]
#        comp    = name.split('_')[2]
#        coor    = name.split('_')[1].upper()
#        label  = f"${prefix}_{comp}$ ({coor})"
#        ax.loglog(med['freq'], med, label=label)
#
#    ax.minorticks_on()
#    ax.set_xlabel('Frequency [Hz]')
#    ax.set_ylabel('Median PSD')
#    ax.set_title(f"Median {str(t0)[11:16]}–{str(t1)[11:16]}  (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)")
#    ax.grid(True, which='both', ls=':')
#    ax.set_yticks([1E-8, 1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
#    ax.legend(ncol=2)
#    ax.set_xlim(1e-2, 32)
#    ax.set_ylim(1e-8, 1e4)
#    plt.tight_layout()
#
#    if outdir and os.path.isdir(outdir):
#        fn = f"median_bs_E_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
#        fig.savefig(os.path.join(outdir, fn), dpi=300, bbox_inches='tight')
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)
#
#plot_median_dict(noise_da_dict, noise_t0, noise_t1, outdir)

In [ ]:
import xarray as xr
import numpy as np

def make_cwt_clean_segments_64(dsets_64, noise_ds):
    """
    入力:
      dsets_64 : [ds_64_fac_cwt_seg0, ...]
      noise_ds: 各成分のノイズ床（frequency or freq 次元, 1D）
    出力:
      cleaned_64: [ds_64_fac_cwt_clean_seg0, ...]
    """
    def _noise_for(var):
        if var not in noise_ds:
            return None
        nda = noise_ds[var]
        if "frequency" in nda.dims:
            nda = nda.rename({"frequency": "freq"})
        return nda

    target_vars = [
        "E64_fac_x_cwt","E64_fac_y_cwt","E64_fac_z_cwt",
        "B64_fac_x_cwt","B64_fac_y_cwt","B64_fac_z_cwt",
    ]

    cleaned_list = []
    for count, ds in enumerate(dsets_64):
        print(f"[seg {count}] dims={ds.dims}")
        # time/freq を持たないセグメントはスキップ
        if ("time" not in ds.coords) or ("freq" not in ds.coords):
            print(f"  -> skip (no time/freq coord)")
            continue

        new_vars = {}
        coords = {"time": ds["time"], "freq": ds["freq"]}

        for v in target_vars:
            if v not in ds:
                continue
            n_da = _noise_for(v)
            if n_da is None:
                continue

            # ノイズ床を各dsのfreqに合わせる
            n_interp = n_da.interp(freq=ds[v].freq)

            cleaned = ds[v] - n_interp
            cleaned = cleaned.where(cleaned > 0)

            new_vars[f"{v}_clean"] = xr.DataArray(
                cleaned.astype(np.float32),
                dims=("time", "freq"),
                coords=coords,
                attrs={**ds[v].attrs, "noise_removed": True}
            )

            coi_name = v.replace("_cwt", "_coi")
            if coi_name in ds:
                new_vars[coi_name] = ds[coi_name]

        cleaned_list.append(xr.Dataset(new_vars, coords=coords))

    return cleaned_list

ds_EB64_fac_cwt_clean_segs = make_cwt_clean_segments_64(ds_EB64_fac_cwt_segs, ds_noise_median)

print(ds_EB64_fac_cwt_clean_segs)

In [ ]:
targets = [
    ("E64_fac_x_cwt_clean", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E64_fac_y_cwt_clean", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E64_fac_z_cwt_clean", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B64_fac_x_cwt_clean", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B64_fac_y_cwt_clean", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B64_fac_z_cwt_clean", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]
joined_EB64_fac_cwt_clean = {}
for v, _, _ in targets:
    da_, coi = concat_cwt_segments(ds_EB64_fac_cwt_clean_segs, v)
    da_ = da_.sortby('freq')
    if da_ is not None: joined_EB64_fac_cwt_clean[v] = (da_.sortby("freq"), coi)

In [ ]:
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330/wavelet_PSD"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#time_windows = [
#    np.datetime64('2022-09-01T21:00') + np.timedelta64(5, 'm')*n
#    for n in range(36)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets):
#        if v not in joined_EB64_fac_cwt_clean: continue
#        da, coi = joined_EB64_fac_cwt_clean[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_clean_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
mu_0 = 4.*np.pi*1E-7

Ex_fac  = xr.concat([ds_EB64_fac_segs[1]['E64_fac_x'], ds_EB64_fac_segs[2]['E64_fac_x'], ds_EB64_fac_segs[3]['E64_fac_x'], ds_EB64_fac_segs[4]['E64_fac_x']], dim='time')
Ey_fac  = xr.concat([ds_EB64_fac_segs[1]['E64_fac_y'], ds_EB64_fac_segs[2]['E64_fac_y'], ds_EB64_fac_segs[3]['E64_fac_y'], ds_EB64_fac_segs[4]['E64_fac_y']], dim='time')
Ez_fac  = xr.concat([ds_EB64_fac_segs[1]['E64_fac_z'], ds_EB64_fac_segs[2]['E64_fac_z'], ds_EB64_fac_segs[3]['E64_fac_z'], ds_EB64_fac_segs[4]['E64_fac_z']], dim='time')
Bx_fac  = xr.concat([ds_EB64_fac_segs[1]['B64_fac_x'], ds_EB64_fac_segs[2]['B64_fac_x'], ds_EB64_fac_segs[3]['B64_fac_x'], ds_EB64_fac_segs[4]['B64_fac_x']], dim='time')
By_fac  = xr.concat([ds_EB64_fac_segs[1]['B64_fac_y'], ds_EB64_fac_segs[2]['B64_fac_y'], ds_EB64_fac_segs[3]['B64_fac_y'], ds_EB64_fac_segs[4]['B64_fac_y']], dim='time')
Bz_fac  = xr.concat([ds_EB64_fac_segs[1]['B64_fac_z'], ds_EB64_fac_segs[2]['B64_fac_z'], ds_EB64_fac_segs[3]['B64_fac_z'], ds_EB64_fac_segs[4]['B64_fac_z']], dim='time')

S_para  = (Ex_fac * By_fac - Ey_fac * Bx_fac) / mu_0 * 1E-12

S_para_toroidal = Ex_fac * By_fac / mu_0 * 1E-12
S_para_poloidal = - Ey_fac * Bx_fac / mu_0 * 1E-12

print(S_para)
print(S_para_toroidal)
print(S_para_poloidal)

In [ ]:
Vph_toroidal    = np.abs(Ey_fac / Bx_fac) * 1E6 # [m/s]
Vph_poloidal    = np.abs(Ex_fac / By_fac) * 1E6 # [m/s]

Vph_perp_comp   = np.sqrt((Ex_fac**2E0 + Ey_fac**2E0) / (Bx_fac**2E0 + By_fac**2E0)) * 1E6 # [m/s]

In [ ]:
import matplotlib as mpl
mpl.rcdefaults()
mpl.rcParams['font.size'] = 20

path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/2230-2330/wavelet_PSD"
)
os.makedirs(path_base_save_plot, exist_ok=True)

targets = [
    ("E64_fac_x_cwt_clean", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E64_fac_y_cwt_clean", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E64_fac_z_cwt_clean", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B64_fac_x_cwt_clean", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B64_fac_y_cwt_clean", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B64_fac_z_cwt_clean", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]

time_windows = [
    (np.datetime64('2022-09-01T22:35:00'),
     np.datetime64('2022-09-01T22:40:00')),

    (np.datetime64('2022-09-01T22:50:30'),
     np.datetime64('2022-09-01T22:53:00')),

    (np.datetime64('2022-09-01T23:09:00'),
     np.datetime64('2022-09-01T23:12:00')),
]

def add_panel_label(ax, label, x=-0.10, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

for t0, t1 in time_windows:
    fig, axes = plt.subplots(len(targets)+2, 1, figsize=(14, 18), sharex=True)
    axes_cwt = axes[:len(targets)]
    for ax, (v, ylab, unit) in zip(axes_cwt, targets):
        if v not in joined_EB64_fac_cwt_clean: continue
        da, coi = joined_EB64_fac_cwt_clean[v]
        plot_cwt_on_ax(ax, da, coi, t0=t0, t1=t1, minutes=None,
                       zrange=(1e-6, 1e3), yrange=(1E-2, 32.0),
                       cmap="turbo", label_left=ylab, unit_right=unit)

    S_para_window   = S_para.sel(time=slice(t0, t1))
    axes[len(targets)].plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    axes[len(targets)].set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    axes[len(targets)].minorticks_on()
    axes[len(targets)].grid(which='both', alpha=0.5)

    Vph_poloidal_window = Vph_poloidal.sel(time=slice(t0, t1))
    Vph_toroidal_window = Vph_toroidal.sel(time=slice(t0, t1))
    Vph_perp_comp_window = Vph_perp_comp.sel(time=slice(t0, t1))
    VA_LEP_window       = ds_velocity_ms_perp_clean['Alfven_speed_LEP'].sel(time=slice(t0, t1))
    VA_MID_window       = ds_velocity_ms_perp_clean['Alfven_speed_MID'].sel(time=slice(t0, t1))
    VA_HFA_window       = ds_velocity_ms_perp_clean['Alfven_speed_HFA'].sel(time=slice(t0, t1))
    axes[len(targets)+1].plot(Vph_perp_comp_window.time, Vph_perp_comp_window.data*1E-3, c='k', lw=0.5, linestyle='solid', label=r'$|\mathbf{E}_{\perp}| / |\mathbf{B}_{\perp}|$')
    #axes[len(targets)+1].plot(Vph_poloidal_window.time, Vph_poloidal_window.data*1E-3, c='green', lw=0.5, linestyle='solid', #label='poloidal')
    #axes[len(targets)+1].plot(Vph_toroidal_window.time, Vph_toroidal_window.data*1E-3, c='orange', lw=0.5, #linestyle='solid', label='toroidal')
    axes[len(targets)+1].plot(VA_LEP_window.time, VA_LEP_window.data*1E-3, c='blue', lw=2, linestyle='dotted', label=r'$v_{\mathrm{A}}$ (LEP)')
    axes[len(targets)+1].plot(VA_MID_window.time, VA_MID_window.data*1E-3, c='k', lw=2, linestyle='dotted', label=r'$v_{\mathrm{A}}$ (MID)')
    axes[len(targets)+1].plot(VA_HFA_window.time, VA_HFA_window.data*1E-3, c='red', lw=2, linestyle='dotted', label=r'$v_{\mathrm{A}}$ (HFA)')
    axes[len(targets)+1].set_ylabel(r'Velocity' + '\n' + r'[$\mathrm{km/s}$]')
    axes[len(targets)+1].set_ylim(ymin=1E3, ymax=1E6)
    axes[len(targets)+1].set_yscale('log')
    axes[len(targets)+1].minorticks_on()
    axes[len(targets)+1].grid(which='both', alpha=0.5)
    axes[len(targets)+1].legend(ncol=5, loc='upper left', fontsize=15)

    axes[-1].set_xlabel("time")
    fig.tight_layout()
    fig.subplots_adjust(hspace=0.1)

    add_panel_label(axes[0], '(1)')
    add_panel_label(axes[1], '(2)')
    add_panel_label(axes[2], '(3)')
    add_panel_label(axes[3], '(4)')
    add_panel_label(axes[4], '(5)')
    add_panel_label(axes[5], '(6)')
    add_panel_label(axes[6], '(7)')
    add_panel_label(axes[7], '(8)')

    if os.path.isdir(path_base_save_plot):
        t0_str = str(t0)
        fn_time = t0_str.replace(':', '').replace('T', '_')
        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_clean_{fn_time}_for_figure.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

# E/B plot

- Poloidal components: $B_{x}$, $E_{y}$
- Toroidal components: $B_{y}$, $E_{x}$

In [ ]:
da_zeros    = xr.zeros_like(joined_EB64_fac_cwt_clean['E64_fac_x_cwt_clean'][0])

ds_EB64_fac_cwt_clean_toroidal  = xr.Dataset({
    'E64_fac_x_cwt_clean':  joined_EB64_fac_cwt_clean['E64_fac_x_cwt_clean'][0],
    'E64_fac_y_cwt_clean':  da_zeros,
    'E64_fac_z_cwt_clean':  da_zeros,
    'B64_fac_x_cwt_clean':  da_zeros,
    'B64_fac_y_cwt_clean':  joined_EB64_fac_cwt_clean['B64_fac_y_cwt_clean'][0],
    'B64_fac_z_cwt_clean':  da_zeros,
})

ds_EB64_fac_cwt_clean_poloidal  = xr.Dataset({
    'E64_fac_x_cwt_clean':  da_zeros,
    'E64_fac_y_cwt_clean':  joined_EB64_fac_cwt_clean['E64_fac_y_cwt_clean'][0],
    'E64_fac_z_cwt_clean':  da_zeros,
    'B64_fac_x_cwt_clean':  joined_EB64_fac_cwt_clean['B64_fac_x_cwt_clean'][0],
    'B64_fac_y_cwt_clean':  da_zeros,
    'B64_fac_z_cwt_clean':  da_zeros,
})

ds_EB64_fac_cwt_clean_perp      = xr.Dataset({
    'E64_fac_x_cwt_clean':  joined_EB64_fac_cwt_clean['E64_fac_x_cwt_clean'][0],
    'E64_fac_y_cwt_clean':  joined_EB64_fac_cwt_clean['E64_fac_y_cwt_clean'][0],
    'E64_fac_z_cwt_clean':  da_zeros,
    'B64_fac_x_cwt_clean':  joined_EB64_fac_cwt_clean['B64_fac_x_cwt_clean'][0],
    'B64_fac_y_cwt_clean':  joined_EB64_fac_cwt_clean['B64_fac_y_cwt_clean'][0],
    'B64_fac_z_cwt_clean':  da_zeros,
})

print(ds_EB64_fac_cwt_clean_toroidal)
print('')
print(ds_EB64_fac_cwt_clean_poloidal)
print('')
print(ds_EB64_fac_cwt_clean_perp)

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'perp'
dsets_64    = ds_EB64_fac_cwt_clean_perp
ds_velocity_ms  = ds_velocity_ms_perp_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdpAx.plot_freq_spectrum_erg(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

#process_and_save_plot(t_min, data_dict, dt_time, out_dir)

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'toroidal'
dsets_64    = ds_EB64_fac_cwt_clean_toroidal
ds_velocity_ms  = ds_velocity_ms_toroidal_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdpAx.plot_freq_spectrum_erg(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

#process_and_save_plot(t_min, data_dict, dt_time, out_dir)

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'poloidal'
dsets_64    = ds_EB64_fac_cwt_clean_poloidal
ds_velocity_ms  = ds_velocity_ms_poloidal_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdpAx.plot_freq_spectrum_erg(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

#process_and_save_plot(t_min, data_dict, dt_time, out_dir)

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'poloidal'
dsets_64    = ds_EB64_fac_cwt_clean_poloidal
ds_velocity_ms  = ds_velocity_ms_poloidal_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T22:35:00', '2022-09-01T22:40:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'toroidal'
dsets_64    = ds_EB64_fac_cwt_clean_toroidal
ds_velocity_ms  = ds_velocity_ms_toroidal

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T22:35:00', '2022-09-01T22:40:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    ## kappa_Bのプロット（エラーバー付き）
    #ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
    #            fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

direction   = 'perp'
dsets_64    = ds_EB64_fac_cwt_clean_perp
ds_velocity_ms  = ds_velocity_ms_perp_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

#time_range = ['2022-09-01T22:35:00', '2022-09-01T22:40:00']
#time_range = ['2022-09-01T22:50:30', '2022-09-01T22:53:00']
time_range = ['2022-09-01T23:09:00', '2022-09-01T23:12:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    #ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
    #            fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    #ax_0.plot(df_results.index, df_results['kappa_B'], marker='.', c='blue', label=r'$\kappa_{\mathrm{B}}$', lw=2)
    #ax_0.axhline(7./3., lw=2, linestyle='dotted', c='blue', label=r'$\kappa_{\mathrm{B}}$ in KAW cascade')
    
    # kappa_Eのプロット（エラーバー付き）
    #ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
    #            fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    ax_0.plot(df_results.index, df_results['kappa_E'], marker='.', c='orange', label=r'$\kappa_{\mathrm{E}}$', lw=0.5)
    ax_0.axhline(1./3., lw=2, linestyle='dotted', c='orange', label=r'$\kappa_{\mathrm{E}}$ in KAW cascade')
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)' + f' ({direction})')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend(loc='best', ncol=4, fontsize=12)
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'poloidal'
dsets_64    = ds_EB64_fac_cwt_clean_poloidal
ds_velocity_ms  = ds_velocity_ms_poloidal_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T22:50:00', '2022-09-01T22:55:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'toroidal'
dsets_64    = ds_EB64_fac_cwt_clean_toroidal
ds_velocity_ms  = ds_velocity_ms_toroidal_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T22:50:00', '2022-09-01T22:55:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'perp'
dsets_64    = ds_EB64_fac_cwt_clean_perp
ds_velocity_ms  = ds_velocity_ms_perp_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T22:50:00', '2022-09-01T22:55:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'poloidal'
dsets_64    = ds_EB64_fac_cwt_clean_poloidal
ds_velocity_ms  = ds_velocity_ms_poloidal_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T23:07:30', '2022-09-01T23:12:30']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'toroidal'
dsets_64    = ds_EB64_fac_cwt_clean_toroidal
ds_velocity_ms  = ds_velocity_ms_toroidal_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T23:07:30', '2022-09-01T23:12:30']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.psd_plotter_Arase_xarray as psdpAx
import importlib
importlib.reload(psdpAx)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

direction   = 'perp'
dsets_64    = ds_EB64_fac_cwt_clean_perp
ds_velocity_ms  = ds_velocity_ms_perp_clean

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 0.4
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01_second/22-24_cleaned_CWT_{dt_time}sec_k_rhoi_{direction}'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------

data_dict = psdpAx.build_data_dict_xr_erg(
    dsets_64, ds_velocity_ms, ds_parameter_clean, cutoff_freq=[1/100, 0.7/8, 5/8, 32]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    t0_np = np.datetime64(pd.Timestamp(t_start).to_datetime64())

    out = psdpAx.plot_k_spectrum_erg(dd, t0_np, dt, n_bins=30)
    if out is None:
        return None
    fig, fit_results = out
    if fig is None:
        return None

    try:
        if isinstance(dt, (float, np.floating)):
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S%f.png')
        else:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

    if not fit_results:
        return None
    if 'time' not in fit_results:
        fit_results = {**fit_results, 'time': pd.Timestamp(t0_np)}
    else:
        fit_results['time'] = pd.Timestamp(fit_results['time'])
    return fit_results

time_range = ['2022-09-01T23:09:00', '2022-09-01T23:12:00']
#time_range = ['2022-09-01T23:07:30', '2022-09-01T23:12:30']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

print(results_list)

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (Arase)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")